In [1]:
%pip install pdfplumber openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 21.0 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 17.2 MB/s eta 0:00:0031m14.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 37.0 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 12.0.0
    Uninstalling pillow-12.0.0:
      Successfully uninstalled pillow-12.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.64.0 requires pillow<13,>=7.1.0, but you have pillow 13.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import pdfplumber

from pathlib import Path

In [5]:
project_dir = (
    Path.home() / "Desktop"
    / "8730 _ Group Assignment"
)

smartcentres_dir = project_dir / "SmartCentres REIT"

riocan_dir = project_dir / "RIO Report"

choice_dir = project_dir / "Choice Properties Annual report"

processed_dir = project_dir / "processed"

processed_dir.mkdir(exist_ok=True)

company_dirs = {
    "SmartCentres": smartcentres_dir,
    "RioCan": riocan_dir,
    "Choice Properties": choice_dir
}

for name, folder in company_dirs.items():
    print(name, "Folder exists:", folder.exists())

SmartCentres Folder exists: True
RioCan Folder exists: True
Choice Properties Folder exists: True


In [7]:
inventory = []

for company, folder in company_dirs.items():

    for file in sorted(folder.rglob("*")):

        if file.is_file() and file.suffix.lower() in [
            ".pdf", ".xlsx", ".xls"
        ]:

            inventory.append({
                "company": company,
                "file_name": file.name,
                "file_type": file.suffix.lower(),
                "file_size_mb": round(
                    file.stat().st_size / 1024**2, 2
                )
            })

financial_inventory = pd.DataFrame(inventory)

display(financial_inventory)

print("Total Files:", len(financial_inventory))

print("\nFiles by Company:")

display(
    financial_inventory.groupby(
        ["company", "file_type"]
    ).size()
)

,company,file_name,file_type,file_size_mb
0,SmartCentres,2023-SmartCentres-REIT-Annual-Report.pdf,.pdf,14.27
1,SmartCentres,2024-SmartCentres-REIT-Annual-Report-.pdf,.pdf,5.16
2,SmartCentres,2025-SmartCentres-REIT-Annual-Report.pdf,.pdf,5.56
3,SmartCentres,2026-Q1-Financial-Statements (2).pdf,.pdf,2.24
4,SmartCentres,2026-Q2-Financial-Statements (2).pdf,.pdf,2.24
5,SmartCentres,2026-Q3-Financial-Statements (1).pdf,.pdf,2.25
6,RioCan,REI-Q4-2024-Report-to-Unitholders.pdf,.pdf,3.58
7,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,.pdf,3.18
8,RioCan,REI_Q4-2023-Annual-Report.pdf,.pdf,5.51
9,RioCan,Q1-2026-Supplemental-Excel.xlsx,.xlsx,0.05


Total Files: 22

Files by Company:


company            file_type
Choice Properties  .pdf         7
RioCan             .pdf         6
                   .xlsx        3
SmartCentres       .pdf         6
dtype: int64

In [9]:
for file in sorted(smartcentres_dir.rglob("*.pdf")):

    print("=" * 60)
    print("File:", file.name)

    with pdfplumber.open(file) as pdf:

        print("Total Pages:", len(pdf.pages))

        text = pdf.pages[0].extract_text()

        print("Text Extraction:", bool(text))

        print("First Page Preview:")

        print(text[:500] if text else "No text found")

File: 2023-SmartCentres-REIT-Annual-Report.pdf
Total Pages: 145
Text Extraction: False
First Page Preview:
No text found
File: 2024-SmartCentres-REIT-Annual-Report-.pdf
Total Pages: 140
Text Extraction: True
First Page Preview:
Canada’s
Shopping Centre
2024 ANNUAL REPORT
File: 2025-SmartCentres-REIT-Annual-Report.pdf
Total Pages: 142
Text Extraction: False
First Page Preview:
No text found
File: 2026-Q1-Financial-Statements (2).pdf
Total Pages: 33
Text Extraction: True
First Page Preview:
Q1
2026 First Quarter Report
Unaudited Interim Condensed Consolidated
Financial Statements
For the three months ended March 31, 2026
File: 2026-Q2-Financial-Statements (2).pdf
Total Pages: 34
Text Extraction: True
First Page Preview:
Q2
2026 Second Quarter Report
Unaudited Interim Condensed Consolidated
Financial Statements
For the three and six months ended June 30, 2026
File: 2026-Q3-Financial-Statements (1).pdf
Total Pages: 34
Text Extraction: True
First Page Preview:
Q 3
2026 Third Quarter Report


In [11]:
# Inspect internal PDF pages

for file in sorted(smartcentres_dir.rglob("*.pdf")):

    print("=" * 60)
    print("File:", file.name)

    with pdfplumber.open(file) as pdf:

        sample_pages = [1, 2, 4, 9, 19]

        for i in sample_pages:

            if i < len(pdf.pages):

                text = pdf.pages[i].extract_text() or ""

                print(
                    "Page:", i + 1,
                    "| Characters:", len(text),
                    "| Readable:", bool(text.strip())
                )

File: 2023-SmartCentres-REIT-Annual-Report.pdf
Page: 2 | Characters: 101 | Readable: True
Page: 3 | Characters: 854 | Readable: True
Page: 5 | Characters: 699 | Readable: True
Page: 10 | Characters: 354 | Readable: True
Page: 20 | Characters: 25686 | Readable: True
File: 2024-SmartCentres-REIT-Annual-Report-.pdf
Page: 2 | Characters: 1056 | Readable: True
Page: 3 | Characters: 651 | Readable: True
Page: 5 | Characters: 829 | Readable: True
Page: 10 | Characters: 20609 | Readable: True
Page: 20 | Characters: 23116 | Readable: True
File: 2025-SmartCentres-REIT-Annual-Report.pdf
Page: 2 | Characters: 618 | Readable: True
Page: 3 | Characters: 1236 | Readable: True
Page: 5 | Characters: 748 | Readable: True
Page: 10 | Characters: 1944 | Readable: True
Page: 20 | Characters: 2336 | Readable: True
File: 2026-Q1-Financial-Statements (2).pdf
Page: 2 | Characters: 61 | Readable: True
Page: 3 | Characters: 428 | Readable: True
Page: 5 | Characters: 1211 | Readable: True
Page: 10 | Characters: 29

In [13]:
# Select the 2025 annual report

target_file = next(
    smartcentres_dir.rglob(
        "2025-SmartCentres-REIT-Annual-Report.pdf"
    )
)

keywords = [
    "rental revenue",
    "net operating income",
    "same property",
    "funds from operations",
    "adjusted funds from operations",
    "occupancy rate",
    "total assets",
    "total liabilities",
    "net asset value",
    "debt to",
    "interest coverage",
    "distributions",
    "gross leasable area"
]

page_texts = {}

with pdfplumber.open(target_file) as pdf:

    for i, page in enumerate(pdf.pages):

        page_texts[i + 1] = page.extract_text() or ""

        page.close()

print("Pages inspected:", len(page_texts))

print("Readable pages:",
      sum(bool(t.strip()) for t in page_texts.values()))

Pages inspected: 142
Readable pages: 138


In [15]:
# Locate financial metrics by keywords

keyword_results = []

for keyword in keywords:

    matched_pages = [
        page_number
        for page_number, text in page_texts.items()
        if keyword.lower() in " ".join(text.lower().split())
    ]

    keyword_results.append({
        "keyword": keyword,
        "matched_pages": matched_pages,
        "number_of_pages": len(matched_pages)
    })

keyword_report = pd.DataFrame(keyword_results)

display(keyword_report)

,keyword,matched_pages,number_of_pages
0,rental revenue,"[32, 51, 108, 109, 114]",5
1,net operating income,"[15, 35, 36, 37, 108]",5
2,same property,"[4, 8]",2
3,funds from operations,"[15, 40]",2
4,adjusted funds from operations,"[15, 40]",2
5,occupancy rate,"[3, 4, 11, 13, 23, 46, 47, 53, 54, 79, 84, 91,...",13
6,total assets,"[3, 46, 55, 59, 70, 95, 113, 138]",8
7,total liabilities,"[55, 95, 113]",3
8,net asset value,"[6, 10, 15, 19, 23, 24, 72]",7
9,debt to,"[11, 15, 16, 18, 63, 70, 138, 139]",8


In [17]:
statement_keywords = [
    "consolidated balance sheets",
    "consolidated statements of financial position",
    "consolidated statements of income",
    "consolidated statements of comprehensive income",
    "consolidated statements of cash flows",
    "management's discussion and analysis",
    "financial highlights",
    "operating highlights"
]

for keyword in statement_keywords:

    pages = [
        p for p, text in page_texts.items()
        if keyword.lower() in " ".join(text.lower().split())
    ]

    print(keyword, ":", pages)

consolidated balance sheets : [55, 88, 89, 95, 100, 101, 126, 128]
consolidated statements of financial position : []
consolidated statements of income : [19, 35, 88, 89, 96, 100, 101, 104, 105, 137, 138]
consolidated statements of comprehensive income : []
consolidated statements of cash flows : [88, 89, 97]
management's discussion and analysis : [91]
financial highlights : [4, 13, 22, 73, 74, 85]
operating highlights : []


In [19]:
# Inspect key financial statements

selected_pages = [95, 96, 97]

for page_number in selected_pages:

    print("=" * 80)
    print("PDF PAGE:", page_number)
    print("=" * 80)

    print(page_texts[page_number])

    print("\n")

PDF PAGE: 95
SMARTCENTRES REAL ESTATE INVESTMENT TRUST
CONSOLIDATED BALANCE SHEETS
(in thousands of Canadian dollars)
As at Note December 31, 2025 December 31, 2024
Assets
Non-current assets
Investment properties 3 $10,852,939 $10,659,783
Equity accounted investments 4 723,637 749,619
Other financial assets 6 94,376 94,974
Mortgages, loans and notes receivable 5 58,740 208,130
Intangible assets 7 39,813 41,145
Amounts receivable 9 17,935 12,994
Other assets 11,825 10,453
$11,799,265 $11,777,098
Current assets
Current portion of mortgages, loans and notes receivable 5 180,070 17,022
Amounts receivable and other 9 74,830 63,441
Cash and cash equivalents 51,551 37,694
Residential development inventory 8 21,154 31,738
Prepaid expenses, deposits and deferred financing costs 9 14,269 12,696
$341,874 $162,591
Total assets $12,141,139 $11,939,689
Liabilities
Non-current liabilities
Debt 10 $4,453,606 $4,059,364
Other financial liabilities 11 17,876 12,619
Other payables 12 11,774 8,158
$4,483,

In [21]:
# Extract financial tables

extracted_tables = {}

with pdfplumber.open(target_file) as pdf:

    for page_number in [95, 96, 97]:

        page = pdf.pages[page_number - 1]

        tables = page.extract_tables()

        print("=" * 60)
        print("PDF Page:", page_number)
        print("Tables Found:", len(tables))

        for i, table in enumerate(tables):

            df = pd.DataFrame(table)

            extracted_tables[(page_number, i)] = df

            print("Table:", i + 1)
            print("Shape:", df.shape)

            display(df)

PDF Page: 95
Tables Found: 1
Table: 1
Shape: (21, 1)


,0
0,"December 31, 2025"
1,
2,"$10,852,939\n723,637\n94,376\n58,740"
3,"39,813"
4,"17,935"
5,"11,825"
6,"$11,799,265"
7,"180,070\n74,830\n51,551\n21,154\n14,269"
8,"$341,874"
9,"$12,141,139"


PDF Page: 96
Tables Found: 1
Table: 1
Shape: (7, 1)


,0
0,2025
1,"$913,913\n(350,871)"
2,"563,042"
3,"(40,612)\n7,716\n(39,292)\n1,053\n(189,106)\n1..."
4,"$310,755"
5,"$251,815\n58,940"
6,"$310,755"


PDF Page: 97
Tables Found: 1
Table: 1
Shape: (9, 1)


,0
0,2025
1,"$310,755\n219,154\n(157,049)\n6,548\n8,604\n(8..."
2,"$377,441"
3,"797,393\n48,224\n10,754\n480,971\n(510,000)\n(..."
4,"$(177,381)"
5,"(40,795)\n(133,651)\n(32,162)\n(1,958)\n(23,80..."
6,"$(186,203)"
7,"13,857\n37,694"
8,"$51,551"


In [23]:
# Inspect REIT-specific performance metrics

performance_pages = [3, 4, 8, 15, 40]

for page_number in performance_pages:

    print("=" * 80)
    print("PDF PAGE:", page_number)
    print("=" * 80)

    print(page_texts[page_number])

    print("\n")

PDF PAGE: 3
SmartCentres
at a Glance
SmartCentres is one of Canada’s largest fully integrated REITs, with a national portfolio of
198 well-located properties serving communities across the country. The portfolio represents
approximately $12.1 billion in assets, including value-oriented retail, purpose-built rental, office,
and self-storage properties. In total, SmartCentres owns 35.6 million square feet of leasable
space with 98.6% committed and in-place occupancy, supported by more than 3,500 acres
of owned land across Canada.
KEY OPERATIONAL
& FINANCIAL STATISTICS
198
SmartCentres’ portfolio includes
ownership interests in retail shopping
centres, mixed-use properties, and Properties at Key
development lands across Canada. Intersections Across Canada
More than 90 percent of the Canadian
population lives within 10 kilometres
of a SmartCentres property.
35.6 MILLION
The portfolio continues to
demonstrate steady financial and
Square Feet of
operational performance, supported
Income-Prod

In [25]:
import re

def parse_financial_number(value):

    value = str(value).strip()

    value = value.replace("$", "")
    value = value.replace(",", "")
    value = value.replace("%", "")

    if value.startswith("(") and value.endswith(")"):
        value = "-" + value[1:-1]

    return float(value)


# Test the function

test_values = [
    "$12,141,139",
    "(350,871)",
    "$(177,381)",
    "98.6%",
    "$0.54"
]

for value in test_values:

    print(value, "->", parse_financial_number(value))

$12,141,139 -> 12141139.0
(350,871) -> -350871.0
$(177,381) -> -177381.0
98.6% -> 98.6
$0.54 -> 0.54


In [27]:
# Extract two annual values from a financial statement row

def extract_metric(text, label):

    matches = []

    for line in text.splitlines():

        line = line.strip()

        if re.match(
            re.escape(label) + r"(?=\s|$)",
            line,
            flags=re.IGNORECASE
        ):

            numbers = re.findall(
                r"\$?\(?-?\d[\d,]*(?:\.\d+)?\)?%?",
                line
            )

            if len(numbers) >= 2:

                matches.append({
                    "original_line": line,
                    "value_2025": parse_financial_number(
                        numbers[-2]
                    ),
                    "value_2024": parse_financial_number(
                        numbers[-1]
                    )
                })

    if len(matches) != 1:
        raise ValueError(
            f"{label}: Expected one row, found {len(matches)}"
        )

    return matches[0]

In [29]:
metric_mapping = {

    # Balance Sheet - Page 95

    "total_assets": (
        95, "Total assets"
    ),

    "total_liabilities": (
        95, "Total liabilities"
    ),

    "non_current_debt": (
        95, "Debt"
    ),

    "current_debt": (
        95, "Current portion of debt"
    ),

    "cash_and_cash_equivalents": (
        95, "Cash and cash equivalents"
    ),

    "investment_properties": (
        95, "Investment properties"
    ),

    # Income Statement - Page 96

    "rental_revenue_and_other": (
        96, "Rentals from investment properties and other"
    ),

    "property_operating_costs": (
        96, "Property operating costs and other"
    ),

    "net_rental_income_and_other": (
        96, "Net rental income and other"
    ),

    "net_income": (
        96, "Net income and comprehensive income"
    ),

    "interest_expense": (
        96, "Interest expense"
    ),

    # Cash Flow Statement - Page 97

    "operating_cash_flow": (
        97, "Cash flows provided by operating activities"
    ),

    "financing_cash_flow": (
        97, "Cash flows used in financing activities"
    ),

    "investing_cash_flow": (
        97, "Cash flows used in investing activities"
    )
}

print("Number of mapped metrics:", len(metric_mapping))

Number of mapped metrics: 14


In [31]:
financial_records = []

for metric, (page_number, label) in metric_mapping.items():

    try:

        result = extract_metric(
            page_texts[page_number],
            label
        )

        for year in [2025, 2024]:

            original_value = result[f"value_{year}"]

            financial_records.append({

                "company": "SmartCentres",

                "ticker": "SRU-UN.TO",

                "period_end": f"{year}-12-31",

                "period_type": "FY",

                "metric": metric,

                "value": original_value / 1000,

                "unit": "CAD million",

                "source_file": target_file.name,

                "source_page": page_number,

                "source_label": label,

                "original_value": original_value,

                "original_unit": "CAD thousand",

                "extraction_method": "Text-based extraction",

                "original_line": result["original_line"]
            })

    except ValueError as e:

        print("REVIEW REQUIRED:", metric, "-", e)


smartcentres_financial = pd.DataFrame(financial_records)

print("Extracted Shape:", smartcentres_financial.shape)

display(smartcentres_financial)

REVIEW REQUIRED: total_liabilities - Total liabilities: Expected one row, found 2
Extracted Shape: (26, 14)


,company,ticker,period_end,period_type,metric,value,unit,source_file,source_page,source_label,original_value,original_unit,extraction_method,original_line
0,SmartCentres,SRU-UN.TO,2025-12-31,FY,total_assets,12141.139,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Total assets,12141139.0,CAD thousand,Text-based extraction,"Total assets $12,141,139 $11,939,689"
1,SmartCentres,SRU-UN.TO,2024-12-31,FY,total_assets,11939.689,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Total assets,11939689.0,CAD thousand,Text-based extraction,"Total assets $12,141,139 $11,939,689"
2,SmartCentres,SRU-UN.TO,2025-12-31,FY,non_current_debt,4453.606,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Debt,4453606.0,CAD thousand,Text-based extraction,"Debt 10 $4,453,606 $4,059,364"
3,SmartCentres,SRU-UN.TO,2024-12-31,FY,non_current_debt,4059.364,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Debt,4059364.0,CAD thousand,Text-based extraction,"Debt 10 $4,453,606 $4,059,364"
4,SmartCentres,SRU-UN.TO,2025-12-31,FY,current_debt,755.426,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Current portion of debt,755426.0,CAD thousand,Text-based extraction,"Current portion of debt 10 755,426 986,915"
5,SmartCentres,SRU-UN.TO,2024-12-31,FY,current_debt,986.915,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Current portion of debt,986915.0,CAD thousand,Text-based extraction,"Current portion of debt 10 755,426 986,915"
6,SmartCentres,SRU-UN.TO,2025-12-31,FY,cash_and_cash_equivalents,51.551,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Cash and cash equivalents,51551.0,CAD thousand,Text-based extraction,"Cash and cash equivalents 51,551 37,694"
7,SmartCentres,SRU-UN.TO,2024-12-31,FY,cash_and_cash_equivalents,37.694,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Cash and cash equivalents,37694.0,CAD thousand,Text-based extraction,"Cash and cash equivalents 51,551 37,694"
8,SmartCentres,SRU-UN.TO,2025-12-31,FY,investment_properties,10852.939,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Investment properties,10852939.0,CAD thousand,Text-based extraction,"Investment properties 3 $10,852,939 $10,659,783"
9,SmartCentres,SRU-UN.TO,2024-12-31,FY,investment_properties,10659.783,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Investment properties,10659783.0,CAD thousand,Text-based extraction,"Investment properties 3 $10,852,939 $10,659,783"


In [33]:
financial_preview = smartcentres_financial.pivot(
    index="metric",
    columns="period_end",
    values="value"
)

display(financial_preview.round(3))

period_end,2024-12-31,2025-12-31
metric,,
cash_and_cash_equivalents,37.694,51.551
current_debt,986.915,755.426
financing_cash_flow,-215.732,-177.381
interest_expense,-183.099,-189.106
investing_cash_flow,-155.525,-186.203
investment_properties,10659.783,10852.939
net_income,292.070,310.755
net_rental_income_and_other,547.508,563.042
non_current_debt,4059.364,4453.606


In [35]:
# Locate additional REIT performance tables

additional_pages = [
    35, 36, 37,
    41, 42,
    46, 47,
    55, 70, 72
]

for page_number in additional_pages:

    print("=" * 80)
    print("PDF PAGE:", page_number)
    print("=" * 80)

    print(page_texts.get(page_number, ""))

    print("\n")

PDF PAGE: 35
MANAGEMENT’S DISCUSSION AND ANALYSIS
Section V — Business Operations and Performance
Results of Operations
Below is a summary of selected financial information concerning the Trust’s operations for the year ended December 31, 2025.
This information should be read in conjunction with the Trust’s consolidated financial statements for the year ended
December 31, 2025.
Proportionately Consolidated Statements of Income and Comprehensive Income (including the Trust’s Interests in Equity
Accounted Investments)
The following tables present the proportionately consolidated statements of income and comprehensive income, which include a
reconciliation of the Trust’s proportionate share of equity accounted investments:
Quarterly Comparison to Prior Year
(in thousands of dollars) Three Months Ended December 31, 2025 Three Months Ended December 31, 2024
Proportionate Total Proportionate Total
Share Proportionate Share Proportionate
GAAP Basis Reconciliation(1) Share(2) GAAP Basis Reconc

In [37]:
def extract_metric(text, label):

    matches = []

    pattern = (
        r"^" + re.escape(label) +
        r"(?=\s+\$?\(?-?\d)"
    )

    for line in text.splitlines():

        line = line.strip()

        if re.match(pattern, line, flags=re.IGNORECASE):

            numbers = re.findall(
                r"\$?\(?-?\d[\d,]*(?:\.\d+)?\)?%?",
                line[len(label):]
            )

            if len(numbers) >= 2:

                matches.append({
                    "original_line": line,
                    "value_2025": parse_financial_number(
                        numbers[-2]
                    ),
                    "value_2024": parse_financial_number(
                        numbers[-1]
                    )
                })

    if len(matches) != 1:

        raise ValueError(
            f"{label}: Expected one row, found {len(matches)}"
        )

    return matches[0]

In [39]:
financial_records = []

for metric, (page_number, label) in metric_mapping.items():

    try:

        result = extract_metric(
            page_texts[page_number],
            label
        )

        for year in [2025, 2024]:

            original_value = result[f"value_{year}"]

            financial_records.append({

                "company": "SmartCentres",

                "ticker": "SRU-UN.TO",

                "period_end": f"{year}-12-31",

                "period_type": "FY",

                "metric": metric,

                "value": original_value / 1000,

                "unit": "CAD million",

                "source_file": target_file.name,

                "source_page": page_number,

                "source_label": label,

                "original_value": original_value,

                "original_unit": "CAD thousand",

                "extraction_method": "Text-based extraction",

                "original_line": result["original_line"]
            })

    except ValueError as e:

        print("REVIEW REQUIRED:", metric, "-", e)


smartcentres_financial = pd.DataFrame(financial_records)

print("Extracted Shape:", smartcentres_financial.shape)

display(smartcentres_financial)

Extracted Shape: (28, 14)


,company,ticker,period_end,period_type,metric,value,unit,source_file,source_page,source_label,original_value,original_unit,extraction_method,original_line
0,SmartCentres,SRU-UN.TO,2025-12-31,FY,total_assets,12141.139,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Total assets,12141139.0,CAD thousand,Text-based extraction,"Total assets $12,141,139 $11,939,689"
1,SmartCentres,SRU-UN.TO,2024-12-31,FY,total_assets,11939.689,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Total assets,11939689.0,CAD thousand,Text-based extraction,"Total assets $12,141,139 $11,939,689"
2,SmartCentres,SRU-UN.TO,2025-12-31,FY,total_liabilities,5794.834,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Total liabilities,5794834.0,CAD thousand,Text-based extraction,"Total liabilities $5,794,834 $5,602,108"
3,SmartCentres,SRU-UN.TO,2024-12-31,FY,total_liabilities,5602.108,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Total liabilities,5602108.0,CAD thousand,Text-based extraction,"Total liabilities $5,794,834 $5,602,108"
4,SmartCentres,SRU-UN.TO,2025-12-31,FY,non_current_debt,4453.606,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Debt,4453606.0,CAD thousand,Text-based extraction,"Debt 10 $4,453,606 $4,059,364"
5,SmartCentres,SRU-UN.TO,2024-12-31,FY,non_current_debt,4059.364,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Debt,4059364.0,CAD thousand,Text-based extraction,"Debt 10 $4,453,606 $4,059,364"
6,SmartCentres,SRU-UN.TO,2025-12-31,FY,current_debt,755.426,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Current portion of debt,755426.0,CAD thousand,Text-based extraction,"Current portion of debt 10 755,426 986,915"
7,SmartCentres,SRU-UN.TO,2024-12-31,FY,current_debt,986.915,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Current portion of debt,986915.0,CAD thousand,Text-based extraction,"Current portion of debt 10 755,426 986,915"
8,SmartCentres,SRU-UN.TO,2025-12-31,FY,cash_and_cash_equivalents,51.551,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Cash and cash equivalents,51551.0,CAD thousand,Text-based extraction,"Cash and cash equivalents 51,551 37,694"
9,SmartCentres,SRU-UN.TO,2024-12-31,FY,cash_and_cash_equivalents,37.694,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,95,Cash and cash equivalents,37694.0,CAD thousand,Text-based extraction,"Cash and cash equivalents 51,551 37,694"


In [41]:
financial_preview = smartcentres_financial.pivot(
    index="metric",
    columns="period_end",
    values="value"
)

display(financial_preview.round(3))

period_end,2024-12-31,2025-12-31
metric,,
cash_and_cash_equivalents,37.694,51.551
current_debt,986.915,755.426
financing_cash_flow,-215.732,-177.381
interest_expense,-183.099,-189.106
investing_cash_flow,-155.525,-186.203
investment_properties,10659.783,10852.939
net_income,292.070,310.755
net_rental_income_and_other,547.508,563.042
non_current_debt,4059.364,4453.606


In [43]:
# Verified REIT performance metrics
# Source: SmartCentres 2025 Annual Report

# Format:
# metric, page, 2025 value, 2024 value, original unit

reit_metrics = [

    ("noi_proportionate", 36,
     592551, 572536, "CAD thousand"),

    ("ffo", 41,
     413838, 402556, "CAD thousand"),

    ("ffo_with_adjustments", 41,
     397922, 383043, "CAD thousand"),

    ("affo", 42,
     369989, 359396, "CAD thousand"),

    ("affo_with_adjustments", 42,
     354073, 339883, "CAD thousand"),

    ("ffo_per_unit_diluted", 42,
     2.27, 2.23, "CAD/unit"),

    ("affo_per_unit_diluted", 42,
     2.03, 1.99, "CAD/unit"),

    ("affo_payout_ratio", 42,
     89.2, 91.7, "Percent"),

    ("occupancy_rate_in_place", 46,
     98.4, 98.2, "Percent"),

    ("occupancy_rate_committed", 46,
     98.6, 98.7, "Percent"),

    ("gross_leasable_area", 46,
     35584637, 35299950, "sq.ft."),

    ("interest_coverage_ratio", 70,
     2.6, 2.5, "Times"),

    ("debt_to_aggregate_assets", 70,
     44.4, 43.7, "Percent"),

    ("adjusted_debt_to_adjusted_ebitda", 70,
     9.7, 9.6, "Times"),

    ("nav_per_unit_diluted", 72,
     35.93, 36.03, "CAD/unit"),

    ("total_distributions_declared", 72,
     330280, 330024, "CAD thousand"),

    ("nav", 72,
     6547534, 6529246, "CAD thousand"),

    ("total_equity", 72,
     6346305, 6337581, "CAD thousand"),

    ("same_property_noi_growth", 4,
     3.7, 2.8, "Percent"),

    ("renewal_rent_growth_including_anchors", 4,
     6.3, 6.1, "Percent")

]

print("REIT Metrics:", len(reit_metrics))

REIT Metrics: 20


In [45]:
reit_records = []

for metric, page, v2025, v2024, original_unit in reit_metrics:

    for year, original_value in [
        (2025, v2025),
        (2024, v2024)
    ]:

        if original_unit == "CAD thousand":

            value = original_value / 1000
            unit = "CAD million"

        else:

            value = original_value
            unit = original_unit

        reit_records.append({

            "company": "SmartCentres",

            "ticker": "SRU-UN.TO",

            "period_end": f"{year}-12-31",

            "period_type": "FY",

            "metric": metric,

            "value": value,

            "unit": unit,

            "source_file": target_file.name,

            "source_page": page,

            "source_label": metric,

            "original_value": original_value,

            "original_unit": original_unit,

            "extraction_method":
                "Manual transcription from extracted text",

            "original_line": None

        })

reit_financial = pd.DataFrame(reit_records)

print("REIT Financial Shape:", reit_financial.shape)

display(reit_financial.head(10))

REIT Financial Shape: (40, 14)


,company,ticker,period_end,period_type,metric,value,unit,source_file,source_page,source_label,original_value,original_unit,extraction_method,original_line
0,SmartCentres,SRU-UN.TO,2025-12-31,FY,noi_proportionate,592.551,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,36,noi_proportionate,592551.0,CAD thousand,Manual transcription from extracted text,None
1,SmartCentres,SRU-UN.TO,2024-12-31,FY,noi_proportionate,572.536,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,36,noi_proportionate,572536.0,CAD thousand,Manual transcription from extracted text,None
2,SmartCentres,SRU-UN.TO,2025-12-31,FY,ffo,413.838,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,41,ffo,413838.0,CAD thousand,Manual transcription from extracted text,None
3,SmartCentres,SRU-UN.TO,2024-12-31,FY,ffo,402.556,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,41,ffo,402556.0,CAD thousand,Manual transcription from extracted text,None
4,SmartCentres,SRU-UN.TO,2025-12-31,FY,ffo_with_adjustments,397.922,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,41,ffo_with_adjustments,397922.0,CAD thousand,Manual transcription from extracted text,None
5,SmartCentres,SRU-UN.TO,2024-12-31,FY,ffo_with_adjustments,383.043,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,41,ffo_with_adjustments,383043.0,CAD thousand,Manual transcription from extracted text,None
6,SmartCentres,SRU-UN.TO,2025-12-31,FY,affo,369.989,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,42,affo,369989.0,CAD thousand,Manual transcription from extracted text,None
7,SmartCentres,SRU-UN.TO,2024-12-31,FY,affo,359.396,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,42,affo,359396.0,CAD thousand,Manual transcription from extracted text,None
8,SmartCentres,SRU-UN.TO,2025-12-31,FY,affo_with_adjustments,354.073,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,42,affo_with_adjustments,354073.0,CAD thousand,Manual transcription from extracted text,None
9,SmartCentres,SRU-UN.TO,2024-12-31,FY,affo_with_adjustments,339.883,CAD million,2025-SmartCentres-REIT-Annual-Report.pdf,42,affo_with_adjustments,339883.0,CAD thousand,Manual transcription from extracted text,None


In [47]:
smartcentres_annual = pd.concat(

    [
        smartcentres_financial,
        reit_financial
    ],

    ignore_index=True

)

smartcentres_annual["period_end"] = pd.to_datetime(
    smartcentres_annual["period_end"]
)

smartcentres_annual = smartcentres_annual.sort_values(
    ["period_end", "metric"]
).reset_index(drop=True)

print("Combined Shape:", smartcentres_annual.shape)

print("\nMetrics per Year:")

display(
    smartcentres_annual.groupby("period_end")["metric"].nunique()
)

display(
    smartcentres_annual.pivot(
        index="metric",
        columns="period_end",
        values="value"
    ).round(3)
)

Combined Shape: (68, 14)

Metrics per Year:


period_end
2024-12-31    34
2025-12-31    34
Name: metric, dtype: int64

period_end,2024-12-31,2025-12-31
metric,,
adjusted_debt_to_adjusted_ebitda,9.600000e+00,9.700000e+00
affo,3.593960e+02,3.699890e+02
affo_payout_ratio,9.170000e+01,8.920000e+01
affo_per_unit_diluted,1.990000e+00,2.030000e+00
affo_with_adjustments,3.398830e+02,3.540730e+02
cash_and_cash_equivalents,3.769400e+01,5.155100e+01
current_debt,9.869150e+02,7.554260e+02
debt_to_aggregate_assets,4.370000e+01,4.440000e+01
ffo,4.025560e+02,4.138380e+02


In [51]:
# Remove previously calculated records
base_data = smartcentres_annual[
    smartcentres_annual["extraction_method"] != "Calculated"
].copy()

# Create derived metrics DataFrame
derived_df = pd.DataFrame(derived_records)

# Standardize column data types before concatenation
column_types = {
    "original_value": "Float64",
    "original_unit": "string",
    "original_line": "string"
}

base_data = base_data.astype(column_types)
derived_df = derived_df.astype(column_types)

# Combine datasets
smartcentres_annual = pd.concat(
    [base_data, derived_df],
    ignore_index=True
)

print("Final Annual Shape:", smartcentres_annual.shape)

Final Annual Shape: (74, 14)


In [53]:
pd.set_option("display.float_format", "{:,.3f}".format)

display(
    smartcentres_annual.pivot(
        index="metric",
        columns="period_end",
        values="value"
    )
)

period_end,2024-12-31,2025-12-31
metric,,
adjusted_debt_to_adjusted_ebitda,9.600,9.700
affo,359.396,369.989
affo_payout_ratio,91.700,89.200
affo_per_unit_diluted,1.990,2.030
affo_with_adjustments,339.883,354.073
book_debt_to_assets_ratio,42.265,42.904
cash_and_cash_equivalents,37.694,51.551
current_debt,986.915,755.426
debt_to_aggregate_assets,43.700,44.400


In [55]:
# Financial consistency validation

for period, group in smartcentres_annual.groupby(
    "period_end"
):

    values = group.set_index("metric")["value"]

    print("=" * 60)
    print("Period:", period.date())

    balance_check = np.isclose(
        values["total_assets"],
        values["total_liabilities"] +
        values["total_equity"],
        atol=0.001,
        rtol=0
    )

    rental_income_check = np.isclose(
        values["rental_revenue_and_other"] +
        values["property_operating_costs"],
        values["net_rental_income_and_other"],
        atol=0.001,
        rtol=0
    )

    debt_check = np.isclose(
        values["current_debt"] +
        values["non_current_debt"],
        values["total_debt"],
        atol=0.001,
        rtol=0
    )

    print("Balance Sheet Check:", balance_check)
    print("Net Rental Income Check:", rental_income_check)
    print("Total Debt Check:", debt_check)


primary_key = [
    "ticker",
    "period_end",
    "period_type",
    "metric"
]

print("\nTotal Records:", len(smartcentres_annual))

print("Duplicate Primary Keys:",
      smartcentres_annual.duplicated(primary_key).sum())

print("Missing Financial Values:",
      smartcentres_annual["value"].isna().sum())

print("\nRecords by Extraction Method:")

display(
    smartcentres_annual.groupby(
        "extraction_method"
    ).size()
)

Period: 2024-12-31
Balance Sheet Check: True
Net Rental Income Check: True
Total Debt Check: True
Period: 2025-12-31
Balance Sheet Check: True
Net Rental Income Check: True
Total Debt Check: True

Total Records: 74
Duplicate Primary Keys: 0
Missing Financial Values: 0

Records by Extraction Method:


extraction_method
Calculated                                   6
Manual transcription from extracted text    40
Text-based extraction                       28
dtype: int64

In [57]:
# Preserve 2024 and 2025 annual data

smartcentres_2024_2025 = smartcentres_annual.copy()

print("Preserved Shape:", smartcentres_2024_2025.shape)

display(
    smartcentres_2024_2025.groupby(
        "period_end"
    )["metric"].nunique()
)

Preserved Shape: (74, 14)


period_end
2024-12-31    37
2025-12-31    37
Name: metric, dtype: int64

In [59]:
# Load SmartCentres 2024 Annual Report

target_file_2024 = next(
    smartcentres_dir.rglob(
        "2024-SmartCentres-REIT-Annual-Report-.pdf"
    )
)

page_texts_2024 = {}

with pdfplumber.open(target_file_2024) as pdf:

    for i, page in enumerate(pdf.pages):

        page_texts_2024[i + 1] = (
            page.extract_text() or ""
        )

        page.close()

print("File:", target_file_2024.name)

print("Total Pages:", len(page_texts_2024))

print(
    "Readable Pages:",
    sum(
        bool(t.strip())
        for t in page_texts_2024.values()
    )
)

File: 2024-SmartCentres-REIT-Annual-Report-.pdf
Total Pages: 140
Readable Pages: 140


In [60]:
# Locate financial statements and REIT metrics

keywords_2024 = [
    "consolidated balance sheets",
    "consolidated statements of income",
    "consolidated statements of cash flows",
    "financial highlights",
    "net operating income",
    "reconciliation of ffo",
    "reconciliation of affo",
    "funds from operations",
    "adjusted funds from operations",
    "quarterly results and trends",
    "occupancy rate",
    "gross leasable area",
    "net asset value",
    "interest coverage",
    "debt to aggregate assets",
    "same property",
    "renewal rent"
]

keyword_results_2024 = []

for keyword in keywords_2024:

    matched_pages = [
        page_number
        for page_number, text in page_texts_2024.items()
        if keyword.lower() in
        " ".join(text.lower().split())
    ]

    keyword_results_2024.append({
        "keyword": keyword,
        "matched_pages": matched_pages,
        "number_of_pages": len(matched_pages)
    })

keyword_report_2024 = pd.DataFrame(
    keyword_results_2024
)

with pd.option_context("display.max_colwidth", None):
    display(keyword_report_2024)

,keyword,matched_pages,number_of_pages
0,consolidated balance sheets,"[52, 53, 76, 77, 86, 87, 92, 93, 96, 97, 104, 105]",12
1,consolidated statements of income,"[16, 17, 32, 33, 76, 77, 86, 87, 98, 99, 100, 101]",12
2,consolidated statements of cash flows,"[86, 87, 94, 95]",4
3,financial highlights,"[8, 9]",2
4,net operating income,"[6, 7, 12, 13, 34, 35]",6
5,reconciliation of ffo,"[38, 39]",2
6,reconciliation of affo,"[36, 37, 38, 39]",4
7,funds from operations,"[36, 37]",2
8,adjusted funds from operations,"[36, 37]",2
9,quarterly results and trends,"[8, 9, 44, 45]",4


In [63]:
# Preserve 2024 and 2025 annual data

smartcentres_2024_2025 = smartcentres_annual.copy()

print("Preserved Shape:", smartcentres_2024_2025.shape)

display(
    smartcentres_2024_2025.groupby(
        "period_end"
    )["metric"].nunique()
)

Preserved Shape: (74, 14)


period_end
2024-12-31    37
2025-12-31    37
Name: metric, dtype: int64

In [65]:
# Load SmartCentres 2024 Annual Report

target_file_2024 = next(
    smartcentres_dir.rglob(
        "2024-SmartCentres-REIT-Annual-Report-.pdf"
    )
)

page_texts_2024 = {}

with pdfplumber.open(target_file_2024) as pdf:

    for i, page in enumerate(pdf.pages):

        page_texts_2024[i + 1] = (
            page.extract_text() or ""
        )

        page.close()

print("File:", target_file_2024.name)

print("Total Pages:", len(page_texts_2024))

print(
    "Readable Pages:",
    sum(
        bool(t.strip())
        for t in page_texts_2024.values()
    )
)

File: 2024-SmartCentres-REIT-Annual-Report-.pdf
Total Pages: 140
Readable Pages: 140


In [66]:
# Locate financial statements and REIT metrics

keywords_2024 = [
    "consolidated balance sheets",
    "consolidated statements of income",
    "consolidated statements of cash flows",
    "financial highlights",
    "net operating income",
    "reconciliation of ffo",
    "reconciliation of affo",
    "funds from operations",
    "adjusted funds from operations",
    "quarterly results and trends",
    "occupancy rate",
    "gross leasable area",
    "net asset value",
    "interest coverage",
    "debt to aggregate assets",
    "same property",
    "renewal rent"
]

keyword_results_2024 = []

for keyword in keywords_2024:

    matched_pages = [
        page_number
        for page_number, text in page_texts_2024.items()
        if keyword.lower() in
        " ".join(text.lower().split())
    ]

    keyword_results_2024.append({
        "keyword": keyword,
        "matched_pages": matched_pages,
        "number_of_pages": len(matched_pages)
    })

keyword_report_2024 = pd.DataFrame(
    keyword_results_2024
)

with pd.option_context("display.max_colwidth", None):
    display(keyword_report_2024)

,keyword,matched_pages,number_of_pages
0,consolidated balance sheets,"[52, 53, 76, 77, 86, 87, 92, 93, 96, 97, 104, 105]",12
1,consolidated statements of income,"[16, 17, 32, 33, 76, 77, 86, 87, 98, 99, 100, 101]",12
2,consolidated statements of cash flows,"[86, 87, 94, 95]",4
3,financial highlights,"[8, 9]",2
4,net operating income,"[6, 7, 12, 13, 34, 35]",6
5,reconciliation of ffo,"[38, 39]",2
6,reconciliation of affo,"[36, 37, 38, 39]",4
7,funds from operations,"[36, 37]",2
8,adjusted funds from operations,"[36, 37]",2
9,quarterly results and trends,"[8, 9, 44, 45]",4


In [69]:
# Step 26 - Inspect 2024 financial statements

selected_pages_2024 = [92, 93, 94, 95]

for page_number in selected_pages_2024:

    print("=" * 80)
    print("PDF PAGE:", page_number)
    print("=" * 80)

    text = page_texts_2024[page_number]

    print("Total Characters:", len(text))
    print(text[:3000])
    print("\n")

PDF PAGE: 92
Total Characters: 12931
MANAGEMENT’S DISCUSSION AND ANALYSIS
MANAGEMENT’S DISCUSSION AND ANALYSIS
SMARTCENTRES REAL ESTATE INVESTMENT TRUST SMARTCENTRES REAL ESTATE INVESTMENT TRUST
Section III — Environmental, Social and Governance
The Trust continues to use the GRESB, a global benchmark for sustainabilityfor real estate companies, as a tool to measure its
CONSOLIDATED BALANCE SHEETS CESOG NprSogOreLsIsD reAlaTtivEeD to S itTs ApeTeErsM. InE2N02T4S, t hOeF T rIuNstC suObMmiEtte Ad iNtsD th iCrdO suMbmPiRssEioHnEtoN thSeI VGREE SINB CanOd Mis Epleased to report that it
((in“ thEouSsanGds o”f C)anadian dollars) (hians t himoupsraonvdesd oitfs C sacnoardei arne ldaotilvlaer st)o the prior year. The Trust also scored above the Benchmark Group Average for Leadership,
Policies, Reporting, Risk Management, Risk Assessment, Tenants and Community, Stakeholder Engagement, and Building
As at Note December 31, 2024 December 31, 2023 January 1, 2023 Certifications.
Note 2024 2023
Asse

In [71]:
%pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 MB 37.6 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.5/22.5 MB 41.8 MB/s eta 0:00:0031m40.3 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [73]:
import pymupdf

doc_2024 = pymupdf.open(target_file_2024)

for page_number in [92, 93, 94, 95]:

    page = doc_2024[page_number - 1]

    text = page.get_text("text")

    print("=" * 80)
    print("PDF PAGE:", page_number)
    print("Total Characters:", len(text))
    print("=" * 80)

    print(text[:3000])

PDF PAGE: 92
Total Characters: 2059
SMARTCENTRES REAL ESTATE INVESTMENT TRUST
CONSOLIDATED BALANCE SHEETS
(in thousands of Canadian dollars)
As at
Note
December 31, 2024
December 31, 2023
January 1, 2023
Assets
Non-current assets
Investment properties
3 
$10,659,783
$10,564,269
$10,286,891
Equity accounted investments
4 
749,619
756,919
680,999
Mortgages, loans and notes receivable
5 
208,130
80,532
238,099
Other financial assets
6 
94,974
152,162
171,807
Other assets
10,453
4,167
4,410
Amounts receivable
9 
12,994
—
—
Intangible assets
7 
41,145
42,476
43,807
$11,777,098
$11,600,525
$11,426,013
Current assets
Assets held for sale
— 
— 
42,321 
Residential development inventory
8 
31,738 
51,719 
40,373 
Current portion of mortgages, loans and notes receivable
5 
17,022 
129,777 
86,593 
Amounts receivable and other
9 
63,441 
73,610 
57,124 
Prepaid expenses, deposits and deferred financing costs
9 
12,696 
15,048 
14,474 
Cash and cash equivalents
37,694 
34,743 
35,255 
$162,591
$30

In [75]:
# Load the 2023 annual report

target_file_2023 = next(
    smartcentres_dir.rglob(
        "2023-SmartCentres-REIT-Annual-Report.pdf"
    )
)

page_texts_2023 = {}

with pdfplumber.open(target_file_2023) as pdf:

    for i, page in enumerate(pdf.pages):

        page_texts_2023[i + 1] = (
            page.extract_text() or ""
        )

        page.close()

print("File:", target_file_2023.name)
print("Total Pages:", len(page_texts_2023))

print(
    "Readable Pages:",
    sum(bool(t.strip())
        for t in page_texts_2023.values())
)

File: 2023-SmartCentres-REIT-Annual-Report.pdf
Total Pages: 145
Readable Pages: 143


In [77]:
# Locate 2023 financial statements

for keyword in keywords_2024:

    pages = [
        p for p, text in page_texts_2023.items()

        if keyword.lower() in
        " ".join(text.lower().split())
    ]

    print(keyword, ":", pages)

consolidated balance sheets : [12, 89]
consolidated statements of income : [12, 89]
consolidated statements of cash flows : [89]
financial highlights : []
net operating income : []
reconciliation of ffo : []
reconciliation of affo : []
funds from operations : []
adjusted funds from operations : []
quarterly results and trends : [12]
occupancy rate : [90]
gross leasable area : []
net asset value : [11]
interest coverage : []
debt to aggregate assets : []
same property : []
renewal rent : []


In [79]:
# Step 29 - Re-extract 2024 report using PyMuPDF

page_texts_2024_clean = {}

with pymupdf.open(target_file_2024) as doc:

    for i, page in enumerate(doc):

        page_texts_2024_clean[i + 1] = (
            page.get_text("text")
        )

print("Total Pages:", len(page_texts_2024_clean))

print("Readable Pages:",
      sum(bool(t.strip())
          for t in page_texts_2024_clean.values()))

Total Pages: 140
Readable Pages: 140


In [81]:
# Step 30 - Extract financial rows with variable year columns

def extract_financial_years(text, label, years):

    matches = []

    number_pattern = (
        r"\$?\(?-?\d[\d,]*(?:\.\d+)?\)?%?"
    )

    pattern = (
        r"^" + re.escape(label) +
        r"(?=\s+\$?\(?-?\d)"
    )

    for line in text.splitlines():

        line = line.strip()

        if re.match(pattern, line, re.IGNORECASE):

            numbers = re.findall(
                number_pattern,
                line[len(label):]
            )

            if len(numbers) >= len(years):

                selected = numbers[-len(years):]

                values = {
                    year: parse_financial_number(value)
                    for year, value in zip(years, selected)
                }

                matches.append({
                    "values": values,
                    "original_line": line
                })

    if len(matches) != 1:

        raise ValueError(
            f"{label}: Expected one row, "
            f"found {len(matches)}"
        )

    return matches[0]

In [83]:
# Step 31 - Test financial extraction

result = extract_financial_years(

    page_texts_2024_clean[92],

    "Total assets",

    [2024, 2023, "opening_2023"]

)

print(result)

ValueError: Total assets: Expected one row, found 0

In [85]:
# Extract text using physical layout

with pymupdf.open(target_file_2024) as doc:

    page = doc[91]

    layout_text = page.get_text(
        "text",
        sort=True
    )

print(layout_text)

SMARTCENTRES REAL ESTATE INVESTMENT TRUST
CONSOLIDATED BALANCE SHEETS
 (in thousands of Canadian dollars)

 As at                                                    Note  December 31, 2024 December 31, 2023  January 1, 2023
  Assets
  Non-current assets
    Investment properties                                      3          $10,659,783        $10,564,269      $10,286,891
     Equity accounted investments                            4             749,619            756,919        680,999
    Mortgages, loans and notes receivable                         5             208,130            80,532        238,099
    Other financial assets                                    6             94,974              152,162           171,807
    Other assets                                                                  10,453                4,167            4,410
    Amounts receivable                                     9               12,994           —         —
     Intangible assets           

In [87]:
result = extract_financial_years(
    layout_text,
    "Total assets",
    [2024, 2023, "opening_2023"]
)

print(result)

{'values': {2024: 11939689.0, 2023: 11905422.0, 'opening_2023': 11702153.0}, 'original_line': 'Total assets                                                                    $11,939,689         $11,905,422        $11,702,153'}


In [89]:
# Step 32 - Layout-based PDF extraction

page_texts_2024_clean = {}

with pymupdf.open(target_file_2024) as doc:

    for i, page in enumerate(doc):

        page_texts_2024_clean[i + 1] = page.get_text(
            "text",
            sort=True
        )

print("Total Pages:", len(page_texts_2024_clean))

print("Readable Pages:",
      sum(bool(t.strip())
          for t in page_texts_2024_clean.values()))

Total Pages: 140
Readable Pages: 140


In [91]:
# Step 33 - Financial metric mapping for 2024 report

metric_mapping_2024 = {

    # Balance Sheet

    "total_assets": (
        92, "Total assets"
    ),

    "total_liabilities": (
        92, "Total liabilities"
    ),

    "non_current_debt": (
        92, "Debt"
    ),

    "current_debt": (
        92, "Current portion of debt"
    ),

    "cash_and_cash_equivalents": (
        92, "Cash and cash equivalents"
    ),

    "investment_properties": (
        92, "Investment properties"
    ),

    "trust_unit_equity": (
        92, "Trust Unit equity"
    ),

    "non_controlling_interests": (
        92, "Non-controlling interests"
    ),

    # Income Statement

    "rental_revenue_and_other": (
        93, "Rentals from investment properties and other"
    ),

    "property_operating_costs": (
        93, "Property operating costs and other"
    ),

    "net_rental_income_and_other": (
        93, "Net rental income and other"
    ),

    "net_income": (
        93, "Net income and comprehensive income"
    ),

    "interest_expense": (
        93, "Interest expense"
    ),

    # Cash Flow Statement

    "operating_cash_flow": (
        94, "Cash flows provided by operating activities"
    ),

    "financing_cash_flow": (
        94, "Cash flows used in financing activities"
    ),

    "investing_cash_flow": (
        94, "Cash flows used in (provided by) investing activities"
    )
}

print("Mapped Metrics:", len(metric_mapping_2024))

Mapped Metrics: 16


In [93]:
# Step 34 - Extract 2023 and 2024 financial data

financial_records_2024 = []

for metric, (page, label) in metric_mapping_2024.items():

    if page == 92:

        years = [2024, 2023, "opening_2023"]

    else:

        years = [2024, 2023]

    try:

        result = extract_financial_years(
            page_texts_2024_clean[page],
            label,
            years
        )

        for year in [2024, 2023]:

            original_value = result["values"][year]

            financial_records_2024.append({

                "company": "SmartCentres",
                "ticker": "SRU-UN.TO",
                "period_end": f"{year}-12-31",
                "period_type": "FY",
                "metric": metric,
                "value": original_value / 1000,
                "unit": "CAD million",
                "source_file": target_file_2024.name,
                "source_page": page,
                "source_label": label,
                "original_value": original_value,
                "original_unit": "CAD thousand",
                "extraction_method": "Text-based extraction",
                "original_line": result["original_line"]

            })

    except ValueError as e:

        print("REVIEW REQUIRED:", metric, "-", e)


smartcentres_from_2024 = pd.DataFrame(
    financial_records_2024
)

smartcentres_from_2024["period_end"] = pd.to_datetime(
    smartcentres_from_2024["period_end"]
)

print("Extracted Shape:", smartcentres_from_2024.shape)

display(
    smartcentres_from_2024.pivot(
        index="metric",
        columns="period_end",
        values="value"
    ).round(3)
)

Extracted Shape: (32, 14)


period_end,2023-12-31,2024-12-31
metric,,
cash_and_cash_equivalents,34.743,37.694
current_debt,605.478,986.915
financing_cash_flow,-332.890,-215.732
interest_expense,-163.741,-183.099
investing_cash_flow,1.525,-155.525
investment_properties,"10,564.269","10,659.783"
net_income,510.103,292.070
net_rental_income_and_other,513.561,547.508
non_controlling_interests,"1,086.970","1,094.751"


In [95]:
# Step 35 - Cross-validate 2024 financial data

old_2024 = smartcentres_2024_2025[
    smartcentres_2024_2025["period_end"].dt.year == 2024
].copy()

new_2024 = smartcentres_from_2024[
    smartcentres_from_2024["period_end"].dt.year == 2024
].copy()

comparison = old_2024[
    ["metric", "value", "unit"]
].merge(

    new_2024[["metric", "value", "unit"]],

    on=["metric", "unit"],
    how="inner",
    suffixes=("_2025_report", "_2024_report"),
    validate="one_to_one"

)

comparison["difference"] = (
    comparison["value_2025_report"] -
    comparison["value_2024_report"]
)

comparison["match"] = np.isclose(

    comparison["value_2025_report"],
    comparison["value_2024_report"],
    atol=0.001,
    rtol=0

)

display(comparison)

print("Compared Metrics:", len(comparison))
print("Matched Metrics:", comparison["match"].sum())
print("Mismatched Metrics:", (~comparison["match"]).sum())

print("All Matched:",
      len(comparison) == 14 and comparison["match"].all())

,metric,value_2025_report,unit,value_2024_report,difference,match
0,cash_and_cash_equivalents,37.694,CAD million,37.694,0.000,True
1,current_debt,986.915,CAD million,986.915,0.000,True
2,financing_cash_flow,-215.732,CAD million,-215.732,0.000,True
3,interest_expense,-183.099,CAD million,-183.099,0.000,True
4,investing_cash_flow,-155.525,CAD million,-155.525,0.000,True
5,investment_properties,"10,659.783",CAD million,"10,659.783",0.000,True
6,net_income,292.070,CAD million,292.070,0.000,True
7,net_rental_income_and_other,547.508,CAD million,547.508,0.000,True
8,non_current_debt,"4,059.364",CAD million,"4,059.364",0.000,True
9,operating_cash_flow,374.208,CAD million,374.208,0.000,True


Compared Metrics: 14
Matched Metrics: 14
Mismatched Metrics: 0
All Matched: True


In [97]:
# Step 36 - Locate REIT performance metrics

keywords_clean = [

    "net operating income",
    "reconciliation of ffo",
    "reconciliation of affo",
    "funds from operations",
    "adjusted funds from operations",
    "occupancy rate",
    "gross leasable area",
    "GLA",
    "net asset value",
    "interest coverage",
    "debt to aggregate assets",
    "same property",
    "same-property",
    "renewal rent",
    "quarterly results and trends",
    "distributions declared"

]

results_clean = []

for keyword in keywords_clean:

    pages = [
        p for p, text in page_texts_2024_clean.items()
        if keyword.lower() in
        " ".join(text.lower().split())
    ]

    results_clean.append({

        "keyword": keyword,
        "matched_pages": pages,
        "number_of_pages": len(pages)

    })

with pd.option_context("display.max_colwidth", None):

    display(pd.DataFrame(results_clean))

,keyword,matched_pages,number_of_pages
0,net operating income,"[6, 13, 34, 106]",4
1,reconciliation of ffo,[38],1
2,reconciliation of affo,"[37, 39]",2
3,funds from operations,"[13, 37]",2
4,adjusted funds from operations,"[13, 37]",2
5,occupancy rate,"[9, 11, 21, 44, 45, 46, 50, 51, 77, 82, 88, 106]",12
6,gross leasable area,"[30, 84]",2
7,GLA,"[2, 3, 4]",3
8,net asset value,"[5, 6, 13, 21, 84]",5
9,interest coverage,"[9, 13, 17, 69, 137, 138]",6


In [99]:
# Step 37 - Export cross-validation evidence

from pathlib import Path

project_dir = (
    Path.home() / "Desktop" / "8730 _ Group Assignment"
)

validation_dir = project_dir / "validation"

validation_dir.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    validation_dir / "smartcentres_2024_cross_validation.csv",
    index=False
)

print("Export Successful:",
      (validation_dir /
       "smartcentres_2024_cross_validation.csv").exists())

Export Successful: True


In [101]:
# Step 38 - Preserve 2023 accounting data

smartcentres_2023_accounting = (
    smartcentres_from_2024[
        smartcentres_from_2024["period_end"].dt.year == 2023
    ].copy()
)

print("2023 Accounting Shape:",
      smartcentres_2023_accounting.shape)

display(
    smartcentres_2023_accounting[
        ["metric", "value", "unit", "source_page"]
    ]
)

2023 Accounting Shape: (16, 14)


,metric,value,unit,source_page
1,total_assets,"11,905.422",CAD million,92
3,total_liabilities,"5,546.118",CAD million,92
5,non_current_debt,"4,394.044",CAD million,92
7,current_debt,605.478,CAD million,92
9,cash_and_cash_equivalents,34.743,CAD million,92
11,investment_properties,"10,564.269",CAD million,92
13,trust_unit_equity,"5,272.334",CAD million,92
15,non_controlling_interests,"1,086.970",CAD million,92
17,rental_revenue_and_other,834.581,CAD million,93
19,property_operating_costs,-321.020,CAD million,93


In [103]:
# Step 39 - Inspect REIT performance tables

selected_pages_2024 = [
    9, 34, 38, 39, 44, 69, 71
]

for page in selected_pages_2024:

    print("=" * 80)
    print("PDF PAGE:", page)
    print("=" * 80)

    print(page_texts_2024_clean[page])

    print("\n")

PDF PAGE: 9
                                                                                       MANAGEMENT’S DISCUSSION AND ANALYSIS

MANAGEMENT’S DISCUSSION AND ANALYSIS
FOR THE YEAR ENDED DECEMBER 31, 2024
Section I — Introduction
About this Management’s Discussion and Analysis

This Management’s Discussion and Analysis (“MD&A”) sets out SmartCentres Real Estate Investment Trust’s (“SmartCentres” or
the “Trust”) business overview and strategic direction, and provides an analysis of the financial performance and financial
condition as at December 31, 2024 and for the year ended December 31, 2024, management’s outlook and the risks facing the
business.

This MD&A should be read in conjunction with the Trust’s audited consolidated financial statements for the years ended
December 31, 2024 and December 31, 2023, and the notes contained therein, and the Trust’s annual information form (“AIF”).
Such consolidated financial statements have been prepared in accordance with International Fi

In [105]:
# Step 40 - Search alternative operating metric names

alternative_keywords = [
    "same asset",
    "same-asset",
    "same store",
    "same-store",
    "spnoi",
    "renewals",
    "leasing spread",
    "rental increase",
    "rental rate",
    "rent growth",
    "rent increase",
    "leasing activity",
    "nav per unit"
]

for keyword in alternative_keywords:

    pages = [
        p for p, text in page_texts_2024_clean.items()
        if keyword.lower() in
        " ".join(text.lower().split())
    ]

    print(f"{keyword}: {pages}")

same asset: []
same-asset: []
same store: []
same-store: []
spnoi: [8, 10, 13, 19]
renewals: [20, 32, 46, 56]
leasing spread: []
rental increase: [101]
rental rate: [9, 46, 53, 77, 88, 106]
rent growth: [11]
rent increase: []
leasing activity: [11, 45]
nav per unit: []


In [107]:
# Step 41.1 - Inspect NOI, FFO and AFFO

for page in [34, 38, 39]:

    print("=" * 80)
    print("PDF PAGE:", page)
    print("=" * 80)

    print(page_texts_2024_clean[page])
    print("\n")

PDF PAGE: 34
 MANAGEMENT’S DISCUSSION AND ANALYSIS



 Net Operating Income
 The following tables summarize NOI, related ratios and recovery ratios, provide additional information, and reflect the Trust’s
 proportionate share of equity accounted investments, the sum of which represent a non-GAAP measure:

 Quarterly Comparison to Prior Year
  (in thousands of dollars)                    Three Months Ended December 31, 2024   Three Months Ended December 31, 2023
                                                                                                                            Variance of
                                           Trust portion       Equity          Total  Trust portion      Equity          Total          Total
                                            excluding  Accounted Proportionate     excluding  Accounted  Proportionate Proportionate
                                              EAI Investments         Share(1)          EAI Investments         Share(1)   

In [109]:
# Step 41.2 - Inspect operating and leverage metrics

for page in [9, 10, 11, 19, 20, 44, 46, 69, 71]:

    print("=" * 80)
    print("PDF PAGE:", page)
    print("=" * 80)

    print(page_texts_2024_clean[page])
    print("\n")

PDF PAGE: 9
                                                                                       MANAGEMENT’S DISCUSSION AND ANALYSIS

MANAGEMENT’S DISCUSSION AND ANALYSIS
FOR THE YEAR ENDED DECEMBER 31, 2024
Section I — Introduction
About this Management’s Discussion and Analysis

This Management’s Discussion and Analysis (“MD&A”) sets out SmartCentres Real Estate Investment Trust’s (“SmartCentres” or
the “Trust”) business overview and strategic direction, and provides an analysis of the financial performance and financial
condition as at December 31, 2024 and for the year ended December 31, 2024, management’s outlook and the risks facing the
business.

This MD&A should be read in conjunction with the Trust’s audited consolidated financial statements for the years ended
December 31, 2024 and December 31, 2023, and the notes contained therein, and the Trust’s annual information form (“AIF”).
Such consolidated financial statements have been prepared in accordance with International Fi

In [111]:
# Step 42 - Calculate 2023 derived financial metrics

values_2023 = (
    smartcentres_2023_accounting
    .set_index("metric")["value"]
)

total_equity_2023 = (
    values_2023["trust_unit_equity"] +
    values_2023["non_controlling_interests"]
)

total_debt_2023 = (
    values_2023["current_debt"] +
    values_2023["non_current_debt"]
)

debt_ratio_2023 = (
    total_debt_2023 /
    values_2023["total_assets"] * 100
)

rental_margin_2023 = (
    values_2023["net_rental_income_and_other"] /
    values_2023["rental_revenue_and_other"] * 100
)

print("Total Equity:", round(total_equity_2023, 3))
print("Total Debt:", round(total_debt_2023, 3))
print("Book Debt-to-Assets Ratio:", round(debt_ratio_2023, 3))
print("Net Rental Income Margin:", round(rental_margin_2023, 3))

Total Equity: 6359.304
Total Debt: 4999.522
Book Debt-to-Assets Ratio: 41.994
Net Rental Income Margin: 61.535


In [113]:
# Step 43 - Validate 2023 accounting data

print(
    "Balance Sheet Check:",
    np.isclose(
        values_2023["total_assets"],
        values_2023["total_liabilities"] +
        total_equity_2023,
        atol=0.001,
        rtol=0
    )
)

print(
    "Net Rental Income Check:",
    np.isclose(
        values_2023["rental_revenue_and_other"] +
        values_2023["property_operating_costs"],
        values_2023["net_rental_income_and_other"],
        atol=0.001,
        rtol=0
    )
)

print(
    "Missing Financial Values:",
    smartcentres_2023_accounting["value"].isna().sum()
)

print(
    "Duplicate Primary Keys:",
    smartcentres_2023_accounting.duplicated(
        ["ticker", "period_end", "period_type", "metric"]
    ).sum()
)

Balance Sheet Check: True
Net Rental Income Check: True
Missing Financial Values: 0
Duplicate Primary Keys: 0


In [115]:
# Step 44 - Map verified REIT metrics from the 2024 report

# Format:
# metric, 2023 value, 2024 value, original unit, PDF page, source label

reit_metrics_2023 = [

    # Operating Performance
    ("noi_proportionate", 560756, 572536,
     "CAD thousand", 10, "NOI"),

    ("spnoi", 537067, 551880,
     "CAD thousand", 10, "SPNOI"),

    ("spnoi_growth", 2.2, 2.8,
     "percent", 10, "Change in SPNOI"),

    ("spnoi_growth_excluding_anchors", 3.3, 4.6,
     "percent", 10, "Change in SPNOI excluding Anchors"),

    # FFO and AFFO
    ("ffo", 400965, 402556,
     "CAD thousand", 38, "FFO"),

    ("ffo_with_adjustments", 376592, 383043,
     "CAD thousand", 38, "FFO with adjustments"),

    ("affo", 354424, 359396,
     "CAD thousand", 39, "AFFO"),

    ("affo_with_adjustments", 330051, 339883,
     "CAD thousand", 39, "AFFO with adjustments"),

    ("ffo_per_unit_diluted", 2.23, 2.23,
     "CAD per unit", 39, "FFO - diluted"),

    ("affo_per_unit_diluted", 1.97, 1.99,
     "CAD per unit", 39, "AFFO - diluted"),

    ("affo_payout_ratio", 93.0, 91.7,
     "percent", 39, "Payout Ratio to AFFO"),

    ("distributions_per_unit", 1.85, 1.85,
     "CAD per unit", 10, "Distributions declared per unit"),

    # Leasing and Occupancy
    ("occupancy_rate_in_place", 98.1, 98.2,
     "percent", 44, "In-place occupancy rate"),

    ("occupancy_rate_committed", 98.5, 98.7,
     "percent", 44, "In-place and committed occupancy rate"),

    ("gross_leasable_area", 35044850, 35299950,
     "sq ft", 44, "Total leasable area"),

    ("tenant_renewal_rate", 83.8, 91.7,
     "percent", 46, "Renewal rate including near completion"),

    ("renewal_rent_growth", 4.0, 6.1,
     "percent", 46, "Change in renewed rental rate including Anchors"),

    ("renewal_rent_growth_excluding_anchors", 5.3, 8.8,
     "percent", 46, "Change in renewed rental rate excluding Anchors"),

    # Debt and Financial Risk
    ("debt_to_aggregate_assets", 43.1, 43.7,
     "percent", 69, "Debt to aggregate assets"),

    ("adjusted_debt_to_adjusted_ebitda", 9.6, 9.6,
     "times", 69, "Adjusted Debt to Adjusted EBITDA"),

    ("interest_coverage_ratio", 2.7, 2.5,
     "times", 69, "Interest coverage ratio"),

    ("weighted_average_interest_rate", 4.15, 3.92,
     "percent", 9, "Weighted average interest rate")
]

print("Mapped REIT Metrics:", len(reit_metrics_2023))

Mapped REIT Metrics: 22


In [117]:
# Step 45 - Standardize REIT performance records

reit_records_2023_2024 = []

for metric, value_2023, value_2024, original_unit, page, label in reit_metrics_2023:

    for year, original_value in [
        (2023, value_2023),
        (2024, value_2024)
    ]:

        if original_unit == "CAD thousand":

            value = original_value / 1000
            unit = "CAD million"

        else:

            value = original_value
            unit = original_unit

        reit_records_2023_2024.append({

            "company": "SmartCentres",
            "ticker": "SRU-UN.TO",
            "period_end": f"{year}-12-31",
            "period_type": "FY",
            "metric": metric,
            "value": value,
            "unit": unit,
            "source_file": target_file_2024.name,
            "source_page": page,
            "source_label": label,
            "original_value": original_value,
            "original_unit": original_unit,
            "extraction_method":
                "Manual transcription from extracted text",
            "original_line": None

        })

reit_from_2024 = pd.DataFrame(
    reit_records_2023_2024
)

reit_from_2024["period_end"] = pd.to_datetime(
    reit_from_2024["period_end"]
)

print("REIT Dataset Shape:", reit_from_2024.shape)

display(
    reit_from_2024.pivot(
        index="metric",
        columns="period_end",
        values="value"
    ).round(3)
)

REIT Dataset Shape: (44, 14)


period_end,2023-12-31,2024-12-31
metric,,
adjusted_debt_to_adjusted_ebitda,9.600,9.600
affo,354.424,359.396
affo_payout_ratio,93.000,91.700
affo_per_unit_diluted,1.970,1.990
affo_with_adjustments,330.051,339.883
debt_to_aggregate_assets,43.100,43.700
distributions_per_unit,1.850,1.850
ffo,400.965,402.556
ffo_per_unit_diluted,2.230,2.230


In [119]:
# Step 46 - Check metric and unit consistency

old_metrics = (
    smartcentres_annual[["metric", "unit"]]
    .drop_duplicates()
)

new_metrics = (
    reit_from_2024[["metric", "unit"]]
    .drop_duplicates()
)

metric_check = old_metrics.merge(

    new_metrics,

    on="metric",
    how="outer",
    suffixes=("_existing", "_new"),
    indicator=True

)

display(metric_check)

,metric,unit_existing,unit_new,_merge
0,adjusted_debt_to_adjusted_ebitda,Times,times,both
1,affo,CAD million,CAD million,both
2,affo_payout_ratio,Percent,percent,both
3,affo_per_unit_diluted,CAD/unit,CAD per unit,both
4,affo_with_adjustments,CAD million,CAD million,both
5,book_debt_to_assets_ratio,Percent,NaN,left_only
6,cash_and_cash_equivalents,CAD million,NaN,left_only
7,current_debt,CAD million,NaN,left_only
8,debt_to_aggregate_assets,Percent,percent,both
9,distributions_per_unit,NaN,CAD per unit,right_only


In [121]:
# Step 47 - Locate NAV and NAV per unit

for page in [5, 6, 13, 21, 84]:

    print("=" * 80)
    print("PDF PAGE:", page)
    print("=" * 80)

    print(page_texts_2024_clean[page])
    print("\n")

PDF PAGE: 5
                  Contents





                                                    Management’s Discussion and Analysis         3


                                                           Section I — Introduction                       5


                                                           Section II — Business Overview,                                                                                                                            17                                                                Strategic Direction and Outlook

                                                            Section III — Environmental, Social and                                                                                                                      18                                                      Governance (“ESG”)


                                                           Section IV — Development Activities          20


                             

In [123]:
# Step 48 - Validate 2023 REIT performance records

smartcentres_2023_reit = reit_from_2024[
    reit_from_2024["period_end"].dt.year == 2023
].copy()

print("Total REIT Records:",
      len(smartcentres_2023_reit))

print("Missing Financial Values:",
      smartcentres_2023_reit["value"].isna().sum())

print("Duplicate Primary Keys:",
      smartcentres_2023_reit.duplicated(
          ["ticker", "period_end",
           "period_type", "metric"]
      ).sum())

display(
    smartcentres_2023_reit[
        ["metric", "value", "unit", "source_page"]
    ]
)

Total REIT Records: 22
Missing Financial Values: 0
Duplicate Primary Keys: 0


,metric,value,unit,source_page
0,noi_proportionate,560.756,CAD million,10
2,spnoi,537.067,CAD million,10
4,spnoi_growth,2.200,percent,10
6,spnoi_growth_excluding_anchors,3.300,percent,10
8,ffo,400.965,CAD million,38
10,ffo_with_adjustments,376.592,CAD million,38
12,affo,354.424,CAD million,39
14,affo_with_adjustments,330.051,CAD million,39
16,ffo_per_unit_diluted,2.230,CAD per unit,39
18,affo_per_unit_diluted,1.970,CAD per unit,39


In [125]:
# Step 49 - Harmonize metric names and units

metric_rename = {
    "spnoi_growth": "same_property_noi_growth",
    "renewal_rent_growth":
        "renewal_rent_growth_including_anchors"
}

unit_rename = {
    "percent": "Percent",
    "times": "Times",
    "CAD per unit": "CAD/unit",
    "sq ft": "sq.ft."
}

reit_from_2024_standardized = reit_from_2024.copy()

reit_from_2024_standardized["metric"] = (
    reit_from_2024_standardized["metric"]
    .replace(metric_rename)
)

reit_from_2024_standardized["unit"] = (
    reit_from_2024_standardized["unit"]
    .replace(unit_rename)
)

print("Standardized Shape:",
      reit_from_2024_standardized.shape)

display(
    reit_from_2024_standardized[
        ["metric", "unit"]
    ].drop_duplicates()
)

Standardized Shape: (44, 14)


,metric,unit
0,noi_proportionate,CAD million
2,spnoi,CAD million
4,same_property_noi_growth,Percent
6,spnoi_growth_excluding_anchors,Percent
8,ffo,CAD million
10,ffo_with_adjustments,CAD million
12,affo,CAD million
14,affo_with_adjustments,CAD million
16,ffo_per_unit_diluted,CAD/unit
18,affo_per_unit_diluted,CAD/unit


In [127]:
# Step 50 - Cross-validate 2024 REIT metrics

existing_2024 = smartcentres_annual[
    pd.to_datetime(
        smartcentres_annual["period_end"]
    ).dt.year == 2024
].copy()

new_2024 = reit_from_2024_standardized[
    reit_from_2024_standardized["period_end"]
    .dt.year == 2024
].copy()

reit_comparison = existing_2024[
    ["metric", "value", "unit"]
].merge(
    new_2024[["metric", "value", "unit"]],
    on="metric",
    how="inner",
    suffixes=("_2025_report", "_2024_report"),
    validate="one_to_one"
)

reit_comparison["difference"] = (
    reit_comparison["value_2025_report"] -
    reit_comparison["value_2024_report"]
)

reit_comparison["unit_match"] = (
    reit_comparison["unit_2025_report"] ==
    reit_comparison["unit_2024_report"]
)

reit_comparison["match"] = (
    np.isclose(
        reit_comparison["value_2025_report"],
        reit_comparison["value_2024_report"],
        atol=0.001,
        rtol=0
    )
    & reit_comparison["unit_match"]
)

display(reit_comparison.round(3))

print("Compared Metrics:", len(reit_comparison))

print("Matched Metrics:",
      reit_comparison["match"].sum())

print("Mismatched Metrics:",
      (~reit_comparison["match"]).sum())

print("All Matched:",
      len(reit_comparison) == 17
      and reit_comparison["match"].all())

,metric,value_2025_report,unit_2025_report,value_2024_report,unit_2024_report,difference,unit_match,match
0,adjusted_debt_to_adjusted_ebitda,9.600,Times,9.600,Times,0.000,True,True
1,affo,359.396,CAD million,359.396,CAD million,0.000,True,True
2,affo_payout_ratio,91.700,Percent,91.700,Percent,0.000,True,True
3,affo_per_unit_diluted,1.990,CAD/unit,1.990,CAD/unit,0.000,True,True
4,affo_with_adjustments,339.883,CAD million,339.883,CAD million,0.000,True,True
5,debt_to_aggregate_assets,43.700,Percent,43.700,Percent,0.000,True,True
6,ffo,402.556,CAD million,402.556,CAD million,0.000,True,True
7,ffo_per_unit_diluted,2.230,CAD/unit,2.230,CAD/unit,0.000,True,True
8,ffo_with_adjustments,383.043,CAD million,383.043,CAD million,0.000,True,True
9,gross_leasable_area,"35,299,950.000",sq.ft.,"35,299,950.000",sq.ft.,0.000,True,True


Compared Metrics: 16
Matched Metrics: 16
Mismatched Metrics: 0
All Matched: False


In [129]:
# Step 51 - Save REIT cross-validation evidence

reit_comparison.to_csv(
    validation_dir /
    "smartcentres_2024_reit_cross_validation.csv",
    index=False
)

print(
    "Export Successful:",
    (
        validation_dir /
        "smartcentres_2024_reit_cross_validation.csv"
    ).exists()
)

Export Successful: True


In [131]:
# Step 52 - Locate detailed NAV calculations

nav_keywords = [
    "net asset value",
    "nav",
    "net asset value per",
    "total unitholders",
    "units classified as liabilities",
    "units outstanding"
]

for keyword in nav_keywords:

    pages = [
        page
        for page, text
        in page_texts_2024_clean.items()
        if keyword.lower()
        in " ".join(text.lower().split())
    ]

    print(f"{keyword}: {pages}")

net asset value: [5, 6, 13, 21, 84]
nav: [13, 20, 21, 80, 84, 98]
net asset value per: [5]
total unitholders: [69]
units classified as liabilities: [10, 15, 38, 43, 63, 69, 70, 71, 84, 94, 99, 100, 120, 121, 125, 126, 128, 137]
units outstanding: [10, 39, 40, 44, 70, 71, 128, 131]


In [133]:
for page in [10, 69]:

    print("=" * 80)
    print("PDF PAGE:", page)
    print("=" * 80)

    print(page_texts_2024_clean[page])

PDF PAGE: 10
MANAGEMENT’S DISCUSSION AND ANALYSIS



                                                    Three Months Ended                        Year Ended

 (in thousands of dollars, except per Unit information)    December 31, 2024  December 31, 2023  December 31, 2024  December 31, 2023
 Financial Information

 Rentals from investment properties and
 other(1)                                               229,743                 211,021             918,359             834,581
 Net income and comprehensive income(1)                    141,850                 14,165            292,070              510,103
 Cash flows provided by operating activities(1)                   122,118             93,745            374,208            330,853
 Net rental income and other(1)                              141,580               128,451            547,508               513,561
 NOI(2)(3)                                                   148,614             136,349             572,536            56

In [135]:
# Step 53 - Standardize 2023 derived metrics

derived_metrics_2023 = [
    (
        "total_equity",
        total_equity_2023,
        "CAD million",
        "trust_unit_equity + non_controlling_interests"
    ),
    (
        "total_debt",
        total_debt_2023,
        "CAD million",
        "current_debt + non_current_debt"
    ),
    (
        "book_debt_to_assets_ratio",
        debt_ratio_2023,
        "Percent",
        "total_debt / total_assets * 100"
    ),
    (
        "net_rental_income_margin",
        rental_margin_2023,
        "Percent",
        "net_rental_income_and_other / rental_revenue_and_other * 100"
    )
]

derived_records_2023 = []

for metric, value, unit, formula in derived_metrics_2023:

    derived_records_2023.append({
        "company": "SmartCentres",
        "ticker": "SRU-UN.TO",
        "period_end": pd.Timestamp("2023-12-31"),
        "period_type": "FY",
        "metric": metric,
        "value": value,
        "unit": unit,
        "source_file": target_file_2024.name,
        "source_page": (
            93 if metric == "net_rental_income_margin"
            else 92
        ),
        "source_label": formula,
        "original_value": None,
        "original_unit": None,
        "extraction_method": "Calculated",
        "original_line": None
    })

smartcentres_2023_derived = pd.DataFrame(
    derived_records_2023
)

print("Derived Metrics Shape:",
      smartcentres_2023_derived.shape)

display(
    smartcentres_2023_derived[
        ["metric", "value", "unit", "source_label"]
    ]
)

Derived Metrics Shape: (4, 14)


,metric,value,unit,source_label
0,total_equity,"6,359.304",CAD million,trust_unit_equity + non_controlling_interests
1,total_debt,"4,999.522",CAD million,current_debt + non_current_debt
2,book_debt_to_assets_ratio,41.994,Percent,total_debt / total_assets * 100
3,net_rental_income_margin,61.535,Percent,net_rental_income_and_other / rental_revenue_a...


In [137]:
len(reit_comparison) == 17


False

In [139]:
# Step 54 - Correct cross-validation condition

expected_metrics = (
    set(existing_2024["metric"]) &
    set(new_2024["metric"])
)

print("Expected Common Metrics:", len(expected_metrics))

print("Compared Metrics:", len(reit_comparison))

print("Matched Metrics:", reit_comparison["match"].sum())

print("Mismatched Metrics:", (~reit_comparison["match"]).sum())

print(
    "All Matched:",
    len(expected_metrics) > 0
    and set(reit_comparison["metric"]) == expected_metrics
    and reit_comparison["match"].all()
)

Expected Common Metrics: 16
Compared Metrics: 16
Matched Metrics: 16
Mismatched Metrics: 0
All Matched: True


In [141]:
# Step 55 - Calculate adjusted NAV and NAV per diluted unit

nav_2023 = 6555875 / 1000
nav_2024 = 6529246 / 1000

diluted_units_2023 = 180108234
diluted_units_2024 = 181205536

nav_per_unit_2023 = (
    nav_2023 * 1000000 / diluted_units_2023
)

nav_per_unit_2024 = (
    nav_2024 * 1000000 / diluted_units_2024
)

print("2023 NAV:", round(nav_2023, 3))
print("2023 NAV per Unit:", round(nav_per_unit_2023, 2))

print("2024 NAV:", round(nav_2024, 3))
print("2024 NAV per Unit:", round(nav_per_unit_2024, 2))

print(
    "2024 NAV per Unit Check:",
    round(nav_per_unit_2024, 2) == 36.03
)

2023 NAV: 6555.875
2023 NAV per Unit: 36.4
2024 NAV: 6529.246
2024 NAV per Unit: 36.03
2024 NAV per Unit Check: True


In [143]:
# Step 56 - Supplement 2023 valuation and distribution metrics

supplementary_metrics = [

    {
        "metric": "nav",
        "value": nav_2023,
        "unit": "CAD million",
        "source_page": 69,
        "source_label":
            "Total Unitholders' equity including Units classified as liabilities",
        "original_value": 6555875,
        "original_unit": "CAD thousand",
        "extraction_method":
            "Manual transcription from extracted text"
    },

    {
        "metric": "nav_per_unit_diluted",
        "value": round(nav_per_unit_2023, 2),
        "unit": "CAD/unit",
        "source_page": 69,
        "source_label":
            "Adjusted NAV / diluted units outstanding (PDF pages 69 and 10)",
        "original_value": None,
        "original_unit": None,
        "extraction_method": "Calculated"
    },

    {
        "metric": "total_distributions_declared",
        "value": 329.639,
        "unit": "CAD million",
        "source_page": 10,
        "source_label": "Distributions declared",
        "original_value": 329639,
        "original_unit": "CAD thousand",
        "extraction_method":
            "Manual transcription from extracted text"
    }
]

supplementary_records = []

for item in supplementary_metrics:

    supplementary_records.append({

        "company": "SmartCentres",
        "ticker": "SRU-UN.TO",
        "period_end": pd.Timestamp("2023-12-31"),
        "period_type": "FY",
        "source_file": target_file_2024.name,
        "original_line": None,
        **item

    })

smartcentres_2023_supplementary = pd.DataFrame(
    supplementary_records
)

print(
    "Supplementary Shape:",
    smartcentres_2023_supplementary.shape
)

display(
    smartcentres_2023_supplementary[
        ["metric", "value", "unit", "source_page"]
    ]
)

Supplementary Shape: (3, 14)


,metric,value,unit,source_page
0,nav,"6,555.875",CAD million,69
1,nav_per_unit_diluted,36.400,CAD/unit,69
2,total_distributions_declared,329.639,CAD million,10


In [145]:
# Step 57 - Combine all 2023 annual financial records

smartcentres_2023_reit = (
    reit_from_2024_standardized[
        reit_from_2024_standardized["period_end"]
        .dt.year == 2023
    ].copy()
)

datasets_2023 = [

    smartcentres_2023_accounting,
    smartcentres_2023_reit,
    smartcentres_2023_derived,
    smartcentres_2023_supplementary

]

smartcentres_2023_final = pd.concat(

    [df.dropna(axis=1, how="all")
     for df in datasets_2023],

    ignore_index=True

).reindex(columns=smartcentres_annual.columns)

smartcentres_2023_final = (
    smartcentres_2023_final
    .sort_values("metric")
    .reset_index(drop=True)
)

print("Final 2023 Shape:",
      smartcentres_2023_final.shape)

display(
    smartcentres_2023_final[
        ["metric", "value", "unit", "source_page"]
    ].round(3)
)

Final 2023 Shape: (45, 14)


,metric,value,unit,source_page
0,adjusted_debt_to_adjusted_ebitda,9.600,Times,69
1,affo,354.424,CAD million,39
2,affo_payout_ratio,93.000,Percent,39
3,affo_per_unit_diluted,1.970,CAD/unit,39
4,affo_with_adjustments,330.051,CAD million,39
5,book_debt_to_assets_ratio,41.994,Percent,92
6,cash_and_cash_equivalents,34.743,CAD million,92
7,current_debt,605.478,CAD million,92
8,debt_to_aggregate_assets,43.100,Percent,69
9,distributions_per_unit,1.850,CAD/unit,10


In [147]:
# Step 58 - Final validation of 2023 annual data

primary_key = [
    "ticker",
    "period_end",
    "period_type",
    "metric"
]

print("Total Records:",
      len(smartcentres_2023_final))

print("Unique Metrics:",
      smartcentres_2023_final["metric"].nunique())

print("Duplicate Primary Keys:",
      smartcentres_2023_final
      .duplicated(primary_key).sum())

print("Missing Financial Values:",
      smartcentres_2023_final["value"].isna().sum())

print("Missing Units:",
      smartcentres_2023_final["unit"].isna().sum())

print("Missing Source Files:",
      smartcentres_2023_final["source_file"].isna().sum())

print("\nRecords by Extraction Method:")

print(
    smartcentres_2023_final[
        "extraction_method"
    ].value_counts()
)

Total Records: 45
Unique Metrics: 45
Duplicate Primary Keys: 0
Missing Financial Values: 0
Missing Units: 0
Missing Source Files: 0

Records by Extraction Method:
extraction_method
Manual transcription from extracted text    24
Text-based extraction                       16
Calculated                                   5
Name: count, dtype: int64


In [149]:
# Step 59 - Compare annual metric coverage

annual_preview = pd.concat(

    [
        smartcentres_2023_final[
            ["period_end", "metric", "value"]
        ],

        smartcentres_annual[
            ["period_end", "metric", "value"]
        ]
    ],

    ignore_index=True

)

annual_preview["period_end"] = pd.to_datetime(
    annual_preview["period_end"]
)

annual_coverage = annual_preview.pivot(
    index="metric",
    columns="period_end",
    values="value"
)

display(annual_coverage.round(3))

print("\nMetrics per Year:")

print(annual_coverage.count())

print("\nMetrics requiring additional data:")

display(
    annual_coverage[
        annual_coverage.isna().any(axis=1)
    ]
)

period_end,2023-12-31,2024-12-31,2025-12-31
metric,,,
adjusted_debt_to_adjusted_ebitda,9.600,9.600,9.700
affo,354.424,359.396,369.989
affo_payout_ratio,93.000,91.700,89.200
affo_per_unit_diluted,1.970,1.990,2.030
affo_with_adjustments,330.051,339.883,354.073
book_debt_to_assets_ratio,41.994,42.265,42.904
cash_and_cash_equivalents,34.743,37.694,51.551
current_debt,605.478,986.915,755.426
debt_to_aggregate_assets,43.100,43.700,44.400



Metrics per Year:
period_end
2023-12-31    45
2024-12-31    37
2025-12-31    37
dtype: int64

Metrics requiring additional data:


period_end,2023-12-31,2024-12-31,2025-12-31
metric,,,
distributions_per_unit,1.850,NaN,NaN
non_controlling_interests,"1,086.970",NaN,NaN
renewal_rent_growth_excluding_anchors,5.300,NaN,NaN
spnoi,537.067,NaN,NaN
spnoi_growth_excluding_anchors,3.300,NaN,NaN
tenant_renewal_rate,83.800,NaN,NaN
trust_unit_equity,"5,272.334",NaN,NaN
weighted_average_interest_rate,4.150,NaN,NaN


In [151]:
# Step 60 - Supplement missing 2024 annual metrics

missing_metrics_2024 = [
    "distributions_per_unit",
    "non_controlling_interests",
    "renewal_rent_growth_excluding_anchors",
    "spnoi",
    "spnoi_growth_excluding_anchors",
    "tenant_renewal_rate",
    "trust_unit_equity",
    "weighted_average_interest_rate"
]

supplement_2024_reit = reit_from_2024_standardized[
    (reit_from_2024_standardized["period_end"].dt.year == 2024)
    &
    (reit_from_2024_standardized["metric"].isin(
        missing_metrics_2024
    ))
].copy()

supplement_2024_accounting = smartcentres_from_2024[
    (pd.to_datetime(
        smartcentres_from_2024["period_end"]
    ).dt.year == 2024)
    &
    (smartcentres_from_2024["metric"].isin(
        missing_metrics_2024
    ))
].copy()

supplement_2024 = pd.concat(
    [
        supplement_2024_reit.dropna(axis=1, how="all"),
        supplement_2024_accounting.dropna(axis=1, how="all")
    ],
    ignore_index=True
).reindex(columns=smartcentres_annual.columns)

print("Supplementary 2024 Shape:", supplement_2024.shape)

display(
    supplement_2024[
        ["metric", "value", "unit", "source_page"]
    ].sort_values("metric")
)

Supplementary 2024 Shape: (8, 14)


,metric,value,unit,source_page
2,distributions_per_unit,1.850,CAD/unit,10
7,non_controlling_interests,"1,094.751",CAD million,92
4,renewal_rent_growth_excluding_anchors,8.800,Percent,46
0,spnoi,551.880,CAD million,10
1,spnoi_growth_excluding_anchors,4.600,Percent,10
3,tenant_renewal_rate,91.700,Percent,46
6,trust_unit_equity,"5,242.830",CAD million,92
5,weighted_average_interest_rate,3.920,Percent,9


In [153]:
# Step 61 - Supplement 2025 equity metrics

equity_metrics_2025 = [
    ("trust_unit_equity", 5227500, "Trust Unit equity"),
    ("non_controlling_interests", 1118805,
     "Non-controlling interests")
]

equity_records_2025 = []

for metric, original_value, label in equity_metrics_2025:

    equity_records_2025.append({
        "company": "SmartCentres",
        "ticker": "SRU-UN.TO",
        "period_end": pd.Timestamp("2025-12-31"),
        "period_type": "FY",
        "metric": metric,
        "value": original_value / 1000,
        "unit": "CAD million",
        "source_file": "2025-SmartCentres-REIT-Annual-Report.pdf",
        "source_page": 95,
        "source_label": label,
        "original_value": original_value,
        "original_unit": "CAD thousand",
        "extraction_method":
            "Manual transcription from extracted text",
        "original_line": None
    })

supplement_2025_equity = pd.DataFrame(
    equity_records_2025
)

print("Supplementary Equity Shape:",
      supplement_2025_equity.shape)

display(supplement_2025_equity[
    ["metric", "value", "unit"]
])

Supplementary Equity Shape: (2, 14)


,metric,value,unit
0,trust_unit_equity,"5,227.500",CAD million
1,non_controlling_interests,"1,118.805",CAD million


In [155]:
# Step 62 - Locate remaining 2025 REIT metrics

keywords_2025 = [
    "distributions declared",
    "spnoi",
    "excluding anchors",
    "renewals",
    "renewal rate",
    "renewed rental rate",
    "weighted average interest rate",
    "weighted average contractual interest rate"
]

keyword_results_2025 = []

for keyword in keywords_2025:

    pages = [
        page
        for page, text in page_texts.items()
        if keyword.lower() in
        " ".join(text.lower().split())
    ]

    keyword_results_2025.append({
        "keyword": keyword,
        "matched_pages": pages,
        "number_of_pages": len(pages)
    })

with pd.option_context("display.max_colwidth", None):

    display(pd.DataFrame(keyword_results_2025))

,keyword,matched_pages,number_of_pages
0,distributions declared,"[20, 43, 44, 46, 72, 129, 132]",7
1,spnoi,"[10, 12, 15, 21]",4
2,excluding anchors,"[4, 8, 11, 12, 13, 15, 39, 49]",8
3,renewals,"[13, 22, 49, 58]",4
4,renewal rate,"[49, 91, 108]",3
5,renewed rental rate,[49],1
6,weighted average interest rate,"[11, 67, 101, 118]",4
7,weighted average contractual interest rate,[],0


In [157]:
# Step 63 - Inspect 2025 financial and operating highlights

selected_pages_2025 = [
    4, 11, 13, 15, 16, 19, 40, 46, 47
]

for page in selected_pages_2025:

    print("=" * 80)
    print("PDF PAGE:", page)
    print("=" * 80)

    print(page_texts[page])
    print("\n")

PDF PAGE: 4
2025 Highlights SmartCentres continued its strong
financial and operational performance
with increased demand for retail space.
Anchor-driven Robust Leasing Self-Storage Townhomes
Growth Pipeline Activity Expansion Closings
Welcomed new locations Resilient leasing momentum Opened three new facilities 98% of the 120 units in
for Walmart, Costco and with an industry leading in Ontario, bringing the phase 1 of Vaughan
Mapletri Supermarket. in place and committed total number of operating Northwest Townhomes
Continued construction of occupancy of 98.6%. self-storage properties to 14. closed, with 33 units closed
a 200,000 sq. ft. Canadian 430,000 sq. ft. of existing Undergoing construction in 2025, contributing
Tire flagship store in Leaside vacant space leased and at four additional sites in $5.0 million to 2025 FFO.
neighbourhood of Toronto 125,000 sq. ft. of new build QC and BC and in process
and secured approval for executed during the year. of obtaining municipal
expansion

In [159]:
# Step 64 - Investigate distribution discrepancies

distribution_review = smartcentres_annual[
    smartcentres_annual["metric"]
    == "total_distributions_declared"
]

with pd.option_context("display.max_colwidth", None):

    display(
        distribution_review[
            [
                "period_end",
                "value",
                "source_file",
                "source_page",
                "source_label",
                "original_value",
                "original_unit",
                "extraction_method"
            ]
        ]
    )

,period_end,value,source_file,source_page,source_label,original_value,original_unit,extraction_method
31,2024-12-31,330.024,2025-SmartCentres-REIT-Annual-Report.pdf,72,total_distributions_declared,"330,024.000",CAD thousand,Manual transcription from extracted text
65,2025-12-31,330.280,2025-SmartCentres-REIT-Annual-Report.pdf,72,total_distributions_declared,"330,280.000",CAD thousand,Manual transcription from extracted text


In [161]:
for page in distribution_review["source_page"].dropna().unique():

    page = int(page)

    print("=" * 80)
    print("PDF PAGE:", page)
    print("=" * 80)

    print(page_texts[page])

PDF PAGE: 72
MANAGEMENT’S DISCUSSION AND ANALYSIS
Distributions
The Board of Trustees is responsible for approving distributions. See also details in the “Determination of Distributions”
subsection in this MD&A.
For the year ended December 31, 2025, the Trust paid $330.3 million in cash distributions (for the year ended December 31, 2024
–$330.0 million in cash distributions). The following table summarizes declared distributions:
Year Ended December 31
(in thousands of dollars) 2025 2024
Distributions declared on:
Trust Units $267,681 $267,630
LP Units 47,696 47,489
Other non-controlling interest 414 365
Distributions on Units classified as equity $315,791 $315,484
Distributions on LP Units classified as liabilities – excluding SmartVMC West 3,764 3,815
Distributions on LP Units classified as liabilities – SmartVMC West 10,725 10,725
Distributions on LP Units classified as liabilities $14,489 $14,540
Total distributions declared $330,280 $330,024
Net Asset Value
The following table pr

In [163]:
# Step 65 - Supplement verified 2025 operating metrics

operating_metrics_2025 = [
    (
        "weighted_average_interest_rate",
        4.00,
        11,
        "Weighted average interest rate"
    ),
    (
        "renewal_rent_growth_excluding_anchors",
        8.4,
        13,
        "Lease renewal rent growth excluding Anchors"
    ),
    (
        "spnoi_growth_excluding_anchors",
        5.6,
        13,
        "Annual Same Properties NOI growth excluding Anchors"
    )
]

operating_records_2025 = []

for metric, value, page, label in operating_metrics_2025:

    operating_records_2025.append({
        "company": "SmartCentres",
        "ticker": "SRU-UN.TO",
        "period_end": pd.Timestamp("2025-12-31"),
        "period_type": "FY",
        "metric": metric,
        "value": value,
        "unit": "Percent",
        "source_file": "2025-SmartCentres-REIT-Annual-Report.pdf",
        "source_page": page,
        "source_label": label,
        "original_value": value,
        "original_unit": "Percent",
        "extraction_method":
            "Manual transcription from extracted text",
        "original_line": None
    })

supplement_2025_operating = pd.DataFrame(
    operating_records_2025
)

print("Supplementary Shape:",
      supplement_2025_operating.shape)

display(
    supplement_2025_operating[
        ["metric", "value", "unit", "source_page"]
    ]
)

Supplementary Shape: (3, 14)


,metric,value,unit,source_page
0,weighted_average_interest_rate,4.000,Percent,11
1,renewal_rent_growth_excluding_anchors,8.400,Percent,13
2,spnoi_growth_excluding_anchors,5.600,Percent,13


In [165]:
# Step 66 - Inspect remaining metric source pages

remaining_pages_2025 = [
    12, 21, 43, 44, 49
]

for page in remaining_pages_2025:

    print("=" * 80)
    print("PDF PAGE:", page)
    print("=" * 80)

    print(page_texts[page])
    print("\n")

PDF PAGE: 12
MANAGEMENT’S DISCUSSION AND ANALYSIS
(6) Total Units outstanding include Trust Units and LP Units (each as defined below), including Units classified as liabilities. LP Units classified as equity in the consolidated financial
statements are presented as non-controlling interests.
(7) Total diluted Units outstanding includes total Units outstanding, vested portion of the deferred units issued pursuant to the Deferred Unit Plan (“DUP”), and vested equity incentive
plan units (“EIP units”) granted pursuant to the Equity Incentive Plan (“EIP”).
Three Months Ended Year Ended
(in thousands of dollars, except per Unit information) December 31, 2025 December 31, 2024 December 31, 2025 December 31, 2024
Financial Information
Rentals from investment properties and other(1) 234,170 229,743 913,913 918,359
Net income and comprehensive income(1) 130,113 141,850 310,755 292,070
Cash flows provided by operating activities(1) 127,344 122,118 377,441 374,208
Net rental income and other(1) 

In [167]:
# Step 67 - Update 2023 NAV per unit source

nav_mask = (
    smartcentres_2023_final["metric"]
    == "nav_per_unit_diluted"
)

nav_updates = {
    "value": 36.40,
    "source_file":
        "2025-SmartCentres-REIT-Annual-Report.pdf",
    "source_page": 11,
    "source_label": "NAV per Unit – diluted",
    "original_value": 36.40,
    "original_unit": "CAD/unit",
    "extraction_method":
        "Manual transcription from extracted text",
    "original_line":
        "NAV per Unit – diluted $35.93 $36.03 $36.40"
}

for column, value in nav_updates.items():

    smartcentres_2023_final.loc[
        nav_mask, column
    ] = value

display(
    smartcentres_2023_final.loc[nav_mask]
)

,company,ticker,period_end,period_type,metric,value,unit,source_file,source_page,source_label,original_value,original_unit,extraction_method,original_line
20,SmartCentres,SRU-UN.TO,2023-12-31,FY,nav_per_unit_diluted,36.400,CAD/unit,2025-SmartCentres-REIT-Annual-Report.pdf,11,NAV per Unit – diluted,36.400,CAD/unit,Manual transcription from extracted text,NAV per Unit – diluted $35.93 $36.03 $36.40


In [169]:
# Step 68 - Locate 2023 distribution reconciliation

distribution_keywords = [
    "total distributions declared",
    "other non-controlling interest",
    "distributions on units classified as equity",
    "distributions on lp units classified as liabilities"
]

for keyword in distribution_keywords:

    pages = [
        page
        for page, text in page_texts_2024_clean.items()
        if keyword.lower()
        in " ".join(text.lower().split())
    ]

    print(f"{keyword}: {pages}")

total distributions declared: [71]
other non-controlling interest: [71, 128, 131]
distributions on units classified as equity: [71, 128]
distributions on lp units classified as liabilities: [71]


In [171]:
# Inspect distribution reconciliation pages

for page in [10, 71, 95]:

    print("=" * 80)
    print("PDF PAGE:", page)
    print("=" * 80)

    print(page_texts_2024_clean[page])
    print("\n")

PDF PAGE: 10
MANAGEMENT’S DISCUSSION AND ANALYSIS



                                                    Three Months Ended                        Year Ended

 (in thousands of dollars, except per Unit information)    December 31, 2024  December 31, 2023  December 31, 2024  December 31, 2023
 Financial Information

 Rentals from investment properties and
 other(1)                                               229,743                 211,021             918,359             834,581
 Net income and comprehensive income(1)                    141,850                 14,165            292,070              510,103
 Cash flows provided by operating activities(1)                   122,118             93,745            374,208            330,853
 Net rental income and other(1)                              141,580               128,451            547,508               513,561
 NOI(2)(3)                                                   148,614             136,349             572,536            56

In [173]:
# Step 69 - Final corrections and supplementary metrics

# Correct 2023 total distributions
mask = (
    smartcentres_2023_final["metric"]
    == "total_distributions_declared"
)

updates = {
    "value": 329.977,
    "original_value": 329977,
    "source_page": 71,
    "source_label": "Total distributions declared",
    "original_line":
        "Total distributions declared $330,024 $329,977"
}

for col, value in updates.items():
    smartcentres_2023_final.loc[mask, col] = value

# Supplement remaining 2025 metrics
remaining_metrics = [
    ("spnoi", 576.630, "CAD million",
     576630, "CAD thousand", "SPNOI", [12, 21, 15, 10]),

    ("tenant_renewal_rate", 88.0, "Percent",
     88.0, "Percent", "Renewal rate", [49])
]

remaining_records = []

for metric, value, unit, original, original_unit, label, pages in remaining_metrics:

    # Locate supporting source page
    number = f"{original:,.0f}"

    matched_pages = [
        p for p in pages
        if number in page_texts[p]
        and label.lower() in page_texts[p].lower()
    ]

    if not matched_pages:
        print("Review required:", metric)
        continue

    remaining_records.append({
        "company": "SmartCentres",
        "ticker": "SRU-UN.TO",
        "period_end": pd.Timestamp("2025-12-31"),
        "period_type": "FY",
        "metric": metric,
        "value": value,
        "unit": unit,
        "source_file":
            "2025-SmartCentres-REIT-Annual-Report.pdf",
        "source_page": matched_pages[0],
        "source_label": label,
        "original_value": original,
        "original_unit": original_unit,
        "extraction_method":
            "Manual transcription from extracted text",
        "original_line": None
    })

supplement_2025_remaining = pd.DataFrame(
    remaining_records,
    columns=smartcentres_annual.columns
)

print("Additional Records:",
      len(supplement_2025_remaining))

display(supplement_2025_remaining[
    ["metric", "value", "unit", "source_page"]
])

Additional Records: 2


,metric,value,unit,source_page
0,spnoi,576.630,CAD million,12
1,tenant_renewal_rate,88.000,Percent,49


In [175]:
# Step 70 - Final annual data integration

annual_datasets = [
    smartcentres_2023_final,
    smartcentres_annual,
    supplement_2024,
    supplement_2025_equity,
    supplement_2025_operating,
    supplement_2025_remaining
]

smartcentres_annual_final = pd.concat(
    [
        df.dropna(axis=1, how="all")
        for df in annual_datasets
        if not df.empty
    ],
    ignore_index=True
).reindex(columns=smartcentres_annual.columns)

# Standardize dates and numeric values
smartcentres_annual_final["period_end"] = pd.to_datetime(
    smartcentres_annual_final["period_end"]
)

smartcentres_annual_final["value"] = pd.to_numeric(
    smartcentres_annual_final["value"],
    errors="coerce"
)

smartcentres_annual_final = (
    smartcentres_annual_final
    .sort_values(["period_end", "metric"])
    .reset_index(drop=True)
)

# Primary key validation
primary_key = [
    "ticker",
    "period_end",
    "period_type",
    "metric"
]

duplicates = smartcentres_annual_final.duplicated(
    primary_key
).sum()

required_columns = primary_key + [
    "value", "unit", "source_file",
    "source_page", "extraction_method"
]

missing = smartcentres_annual_final[
    required_columns
].isna().sum().sum()

unit_conflicts = (
    smartcentres_annual_final
    .groupby("metric")["unit"]
    .nunique()
    .gt(1)
    .sum()
)

print("Final Shape:",
      smartcentres_annual_final.shape)

print("Duplicate Primary Keys:", duplicates)
print("Missing Required Values:", missing)
print("Unit Conflicts:", unit_conflicts)

print("\nRecords per Year:")

print(
    smartcentres_annual_final
    .groupby("period_end")["metric"]
    .count()
)

# Check data coverage
final_coverage = smartcentres_annual_final.pivot(
    index="metric",
    columns="period_end",
    values="value"
)

print("\nMissing Metric Coverage:")

display(
    final_coverage[
        final_coverage.isna().any(axis=1)
    ]
)

print(
    "\nStructural Validation:",
    duplicates == 0
    and missing == 0
    and unit_conflicts == 0
)

Final Shape: (134, 14)
Duplicate Primary Keys: 0
Missing Required Values: 0
Unit Conflicts: 0

Records per Year:
period_end
2023-12-31    45
2024-12-31    45
2025-12-31    44
Name: metric, dtype: int64

Missing Metric Coverage:


period_end,2023-12-31,2024-12-31,2025-12-31
metric,,,
distributions_per_unit,1.850,1.850,NaN



Structural Validation: True


In [177]:
# Step 71 - Export final annual financial dataset

from pathlib import Path

processed_dir = (
    Path.home()
    / "Desktop"
    / "8730 _ Group Assignment"
    / "processed"
)

processed_dir.mkdir(parents=True, exist_ok=True)

output_file = (
    processed_dir /
    "smartcentres_annual_financials_cleaned.csv"
)

assert duplicates == 0, "Duplicate primary keys"
assert missing == 0, "Missing required values"
assert unit_conflicts == 0, "Inconsistent units"

smartcentres_annual_final.to_csv(
    output_file,
    index=False
)

# Verify exported file
saved_data = pd.read_csv(output_file)

print("Export Successful:", output_file.exists())
print("Saved Shape:", saved_data.shape)
print("Saved to:", output_file)

print(
    "Export Validation:",
    saved_data.shape == smartcentres_annual_final.shape
)

Export Successful: True
Saved Shape: (134, 14)
Saved to: /Users/yuanquan0901/Desktop/8730 _ Group Assignment/processed/smartcentres_annual_financials_cleaned.csv
Export Validation: True


In [179]:
# Step 72 - Load quarterly financial reports

from pathlib import Path
import pdfplumber

project_dir = (
    Path.home()
    / "Desktop"
    / "8730 _ Group Assignment"
)

quarterly_files = {}

for quarter in ["Q1", "Q2", "Q3"]:

    matches = list(
        project_dir.rglob(
            f"2026-{quarter}-Financial-Statements*.pdf"
        )
    )

    if len(matches) == 1:
        quarterly_files[quarter] = matches[0]
    else:
        print(quarter, "Files found:", matches)

quarterly_texts = {}

for quarter, file in quarterly_files.items():

    with pdfplumber.open(file) as pdf:

        quarterly_texts[quarter] = {
            i + 1: page.extract_text() or ""
            for i, page in enumerate(pdf.pages)
        }

    print("=" * 60)
    print("Quarter:", quarter)
    print("File:", file.name)
    print("Total Pages:", len(quarterly_texts[quarter]))

    for page, text in quarterly_texts[quarter].items():

        normalized = " ".join(text.lower().split())

        keywords = [
            "total assets",
            "total liabilities",
            "rentals from investment properties",
            "net income and comprehensive income"
        ]

        matched = [
            k for k in keywords
            if k in normalized
        ]

        if matched:
            print("Page:", page, "|", matched)

Quarter: Q1
File: 2026-Q1-Financial-Statements (2).pdf
Total Pages: 33
Page: 4 | ['total assets', 'total liabilities']
Page: 5 | ['rentals from investment properties']
Page: 7 | ['net income and comprehensive income']
Page: 12 | ['total assets', 'total liabilities']
Page: 26 | ['rentals from investment properties']
Page: 32 | ['rentals from investment properties']
Quarter: Q2
File: 2026-Q2-Financial-Statements (2).pdf
Total Pages: 34
Page: 4 | ['total assets', 'total liabilities']
Page: 5 | ['rentals from investment properties']
Page: 7 | ['net income and comprehensive income']
Page: 12 | ['total assets', 'total liabilities']
Page: 27 | ['rentals from investment properties']
Page: 33 | ['rentals from investment properties']
Quarter: Q3
File: 2026-Q3-Financial-Statements (1).pdf
Total Pages: 34
Page: 4 | ['total assets', 'total liabilities']
Page: 5 | ['rentals from investment properties', 'net income and comprehensive income']
Page: 6 | ['net income and comprehensive income']
Page: 7 |

In [181]:
# Step 73 - Inspect quarterly financial statements

for quarter, pages in quarterly_texts.items():

    for page in [4, 5]:

        print("=" * 80)
        print(f"2026 {quarter} | PDF PAGE: {page}")
        print("=" * 80)

        print(pages[page])
        print("\n")

2026 Q1 | PDF PAGE: 4
SMARTCENTRES REAL ESTATE INVESTMENT TRUST
UNAUDITED INTERIM CONDENSED CONSOLIDATED BALANCE SHEETS
(in thousands of Canadian dollars)
As at Note March 31, 2026 December 31, 2025
Assets
Non-current assets
Investment properties 3 $10,964,730 $10,852,939
Equity accounted investments 4 721,995 723,637
Mortgages, loans and notes receivable 5 244,263 58,740
Other financial assets 6 58,338 94,376
Intangible assets 39,480 39,813
Other assets 11,271 11,825
Amounts receivable 7 2,302 17,935
$12,042,379 $11,799,265
Current assets
Amounts receivable and other 7 103,323 74,830
Cash and cash equivalents 36,513 51,551
Prepaid expenses, deposits and deferred financing costs 7 21,567 14,269
Residential development inventory 20,705 21,154
Current portion of mortgages, loans and notes receivable 5 17,271 180,070
Current portion of other financial assets 6 44,141 —
$243,520 $341,874
Total assets $12,285,899 $12,141,139
Liabilities
Non-current liabilities
Debt 8 $4,129,046 $4,453,606
O

In [183]:
# Step 74 - Quarterly extraction, validation and export

import pandas as pd
import numpy as np

# Financial metrics from PDF pages 4 and 5
# Original amounts are in CAD thousand

quarterly_metrics = [
    ("total_assets", 4,
     12285899, 12062407, 12161614),

    ("total_liabilities", 4,
     5888561, 5890911, 5889491),

    ("total_equity", 4,
     6397338, 6171496, 6272123),

    ("current_debt", 4,
     1128256, 1458253, 1398958),

    ("non_current_debt", 4,
     4129046, 3776132, 3893747),

    ("cash_and_cash_equivalents", 4,
     36513, 44891, 37190),

    ("investment_properties", 4,
     10964730, 10836846, 10896197),

    ("rental_revenue_and_other", 5,
     231835, 225959, 220660),

    ("property_operating_costs", 5,
     -94148, -86024, -78813),

    ("net_rental_income_and_other", 5,
     137687, 139935, 141847),

    ("net_income", 5,
     129897, -146979, 179477),

    ("interest_expense", 5,
     -48940, -48889, -49690)
]

quarter_dates = {
    "Q1": "2026-03-31",
    "Q2": "2026-06-30",
    "Q3": "2026-09-30"
}

quarterly_records = []

for metric, page, q1, q2, q3 in quarterly_metrics:

    for quarter, original in zip(
        ["Q1", "Q2", "Q3"], [q1, q2, q3]
    ):

        quarterly_records.append({
            "company": "SmartCentres",
            "ticker": "SRU-UN.TO",
            "period_end": pd.Timestamp(
                quarter_dates[quarter]
            ),
            "period_type": "Quarterly",
            "metric": metric,
            "value": original / 1000,
            "unit": "CAD million",
            "source_file":
                quarterly_files[quarter].name,
            "source_page": page,
            "source_label": metric,
            "original_value": original,
            "original_unit": "CAD thousand",
            "extraction_method":
                "Manual transcription from extracted text",
            "original_line": None
        })

smartcentres_quarterly = pd.DataFrame(
    quarterly_records
)

smartcentres_quarterly = (
    smartcentres_quarterly
    .sort_values(["period_end", "metric"])
    .reset_index(drop=True)
)

# Data validation

primary_key = [
    "ticker", "period_end",
    "period_type", "metric"
]

required_columns = primary_key + [
    "value", "unit", "source_file",
    "source_page", "extraction_method"
]

duplicates = smartcentres_quarterly.duplicated(
    primary_key
).sum()

missing = smartcentres_quarterly[
    required_columns
].isna().sum().sum()

quarterly_preview = smartcentres_quarterly.pivot(
    index="metric",
    columns="period_end",
    values="value"
)

balance_checks = []
rental_checks = []

for period in quarterly_preview.columns:

    v = quarterly_preview[period]

    balance_check = np.isclose(
        v["total_assets"],
        v["total_liabilities"] + v["total_equity"],
        atol=0.001,
        rtol=0
    )

    rental_check = np.isclose(
        v["rental_revenue_and_other"]
        + v["property_operating_costs"],
        v["net_rental_income_and_other"],
        atol=0.001,
        rtol=0
    )

    balance_checks.append(balance_check)
    rental_checks.append(rental_check)

    print("=" * 60)
    print("Period:", period.date())
    print("Balance Sheet Check:", balance_check)
    print("Net Rental Income Check:", rental_check)

# Final validation

assert smartcentres_quarterly.shape == (36, 14)
assert duplicates == 0
assert missing == 0
assert all(balance_checks)
assert all(rental_checks)

print("\nFinal Shape:",
      smartcentres_quarterly.shape)

print("Duplicate Primary Keys:", duplicates)
print("Missing Required Values:", missing)

display(quarterly_preview.round(3))

# Export CSV

output_file = (
    processed_dir /
    "smartcentres_quarterly_financials_cleaned.csv"
)

smartcentres_quarterly.to_csv(
    output_file,
    index=False
)

saved_data = pd.read_csv(output_file)

print("\nExport Successful:", output_file.exists())
print("Saved Shape:", saved_data.shape)
print("Saved to:", output_file)
print("Export Validation:",
      saved_data.shape == (36, 14))

Period: 2026-03-31
Balance Sheet Check: True
Net Rental Income Check: True
Period: 2026-06-30
Balance Sheet Check: True
Net Rental Income Check: True
Period: 2026-09-30
Balance Sheet Check: True
Net Rental Income Check: True

Final Shape: (36, 14)
Duplicate Primary Keys: 0
Missing Required Values: 0


period_end,2026-03-31,2026-06-30,2026-09-30
metric,,,
cash_and_cash_equivalents,36.513,44.891,37.190
current_debt,"1,128.256","1,458.253","1,398.958"
interest_expense,-48.940,-48.889,-49.690
investment_properties,"10,964.730","10,836.846","10,896.197"
net_income,129.897,-146.979,179.477
net_rental_income_and_other,137.687,139.935,141.847
non_current_debt,"4,129.046","3,776.132","3,893.747"
property_operating_costs,-94.148,-86.024,-78.813
rental_revenue_and_other,231.835,225.959,220.660



Export Successful: True
Saved Shape: (36, 14)
Saved to: /Users/yuanquan0901/Desktop/8730 _ Group Assignment/processed/smartcentres_quarterly_financials_cleaned.csv
Export Validation: True
